In [107]:
#%pip install scikit-learn

In [108]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append(r"Z:\HTOC\Data_Analytics\threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"H:\HTOC\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    display(f"Loaded config from: {config_path}")
    display(f"Base URL: {api_base_url}")
    display(f"Access ID: {api_access_id}")
    display(f"Default Org: {api_default_org}")
except Exception as e:
    display(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    display("ThreatConnect initialized.")
except Exception as e:
    display(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    display("RequestObject successfully created.")
except Exception as e:
    display(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)




'Loaded config from: H:\\HTOC\\scripts\\Data Movement\\ThrearConnect-api-pull\\utils\\config.json'

'Base URL: https://hvs.threatconnect.com/api'

'Access ID: 09783848890162390382'

'Default Org: HTOC Org'

'ThreatConnect initialized.'

'RequestObject successfully created.'

In [109]:
import pandas as pd
from datetime import datetime, timedelta
import pytz
import urllib.parse

# Configuration for ThreatConnect indicator query
QUERY_LOOKBACK_DAYS = 360  # days of lastObserved activity to include
INDICATOR_TYPE_NAMES = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "User Agent",
]
OWNER_NAMES = [
    'HTOC Org',
    'CISA Federal Feed',
    'CMS_CTI',
    'Crowdstrike Falcon Intelligence',
    'DHS CISCP',
    'Intel471',
    'Mandiant Advantage Threat Intelligence',
    'VA_TIP Data',
]
RESULT_PAGE_SIZE = 500  # keep this smaller; same fields, just paged

# Setup
cutoff = pd.Timestamp.utcnow()
start_date = (datetime.now(pytz.UTC) - timedelta(days=QUERY_LOOKBACK_DAYS)).date()
start = f"{start_date}T00:00:00Z"

type_names = INDICATOR_TYPE_NAMES
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

list_of_owners = OWNER_NAMES

# Build owner IN (...) clause
owner_condition = ", ".join([f'"{o}"' for o in list_of_owners])

tql_raw = (
    f'ownerName IN ({owner_condition}) AND '
    f'typeName IN ({type_name_condition}) AND '
    f'lastObserved >= "{start}"'
)

tql_encoded = urllib.parse.quote(tql_raw)

final_results = []

# Query indicators (paginate so you don't 502 with heavy fields)
# Create a NEW RequestObject WITHOUT owner restriction to query across all owners
ro_multi = RequestObject()
ro_multi.set_http_method('GET')

result_start = 0
result_limit = RESULT_PAGE_SIZE

while True:
    try:
        # NOTE: same fields list you requested (tags,observations,associatedGroups,falsePositives,threatAssess)
        # Only change here is removing the trailing comma after threatAssess which can break parsing.
        ro_multi.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}'
            f'&fields=tags,observations,associatedGroups,falsePositives,threatAssess'
            f'&resultStart={result_start}&resultLimit={result_limit}'
        )

        response = tc.api_request(ro_multi)

        ct = response.headers.get('content-type', '')
        if not ct.startswith('application/json'):
            raise RuntimeError(f"Non-JSON response ({ct}): {response.content[:200]}")

        results = response.json()
        data_items = results.get('data', []) or []

        # stop when no more results
        if not data_items:
            break

        final_results.append(results)
        result_start += result_limit

    except Exception as e:
        display(f"Failed to query indicators (start={result_start}): {e}")
        break

# Normalize results
normalized_data = []
for result in final_results:
    data_items = result.get('data', [])
    if not data_items:
        display("No data returned in API response:", result)
    for item in data_items:
        if isinstance(item, dict) and 'summary' in item:
            normalized_data.append(item)

if normalized_data:
    observed_src = pd.json_normalize(normalized_data)
    observed_src['indicator'] = observed_src['summary'].astype(str).str.split().str[0].str.strip()
    observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True, errors='coerce')
    observed_src = observed_src[observed_src['lastObserved'] >= pd.to_datetime(start, utc=True)]
    
    # Create a 'sources' column by aggregating ownerName values per indicator
    sources_per_indicator = (
        observed_src.groupby('indicator')['ownerName']
        .apply(lambda x: ', '.join(sorted(set(x))))
        .reset_index()
        .rename(columns={'ownerName': 'sources'})
    )

    # Merge sources back into observed_src
    observed_src = observed_src.merge(sources_per_indicator, on='indicator', how='left')
    # Filter to keep only records where ownerName is 'HTOC Org'
    observed_src = observed_src[observed_src['ownerName'] == 'HTOC Org'].copy()
else:
    display("No valid indicator data found.")
    observed_src = pd.DataFrame()

display(observed_src)

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,hostName,dnsActive,whoisActive,address,text,lastFalsePositive,sha256,md5,indicator,sources
0,6755399459078631,2025-06-25T17:45:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:11:02Z,3.0,46.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.236,HTOC Org
1,6755399459078632,2025-06-25T17:45:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:51Z,3.0,41.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.53,HTOC Org
2,6755399460003050,2025-06-30T12:21:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:39Z,3.0,40.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.199,HTOC Org
3,6755399460003051,2025-06-30T12:21:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:28Z,3.0,45.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.62.156.97,HTOC Org
4,6755399460003056,2025-06-30T12:21:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:16Z,3.0,42.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.49.1.175,HTOC Org
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6145,1682885,2019-06-18T15:13:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-10T23:23:45Z,NaN,NaN,3.86,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178.162.203.226,HTOC Org
6146,4291925,2023-01-24T03:34:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-09T12:17:54Z,4.0,0.0,3.50,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,45.88.221.211,HTOC Org
6149,4329148,2023-05-03T14:27:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-15T15:42:23Z,3.0,0.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,178.175.129.37,HTOC Org
6150,4501780,2024-01-18T19:06:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-13T11:09:41Z,3.0,0.0,3.00,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,185.44.77.71,HTOC Org


In [110]:
import pandas as pd

# Configuration for observed tags
TAGS_FILE_PATH = r"Z:\HTOC\Data_Analytics\Data\Observed_Tags\htoc_observed_indicator_tags.csv"
THREAT_CATEGORY_FILTER = 'THREAT ACTOR'

# Load the observed tags data from the CSV file
tags_file_path = TAGS_FILE_PATH
htoc_observed_tags = pd.read_csv(tags_file_path)

# Filter for THREAT ACTOR category only
threat_actor_records = htoc_observed_tags[
    htoc_observed_tags['threat_category'] == THREAT_CATEGORY_FILTER
]

# Group by indicator and aggregate the tag information
threat_actor_condensed = threat_actor_records.groupby('indicator').agg({
    'type': 'first',
    'orig_tag': lambda x: ', '.join(sorted(set(x.dropna()))),
    'tag': lambda x: ', '.join(sorted(set(x.dropna()))),
    'threat_category': lambda x: ', '.join(sorted(set(x.dropna()))),
    'NATION STATE': lambda x: ', '.join(sorted(set(x.dropna()))) if x.notna().any() else None,
    'SECURITY ORGANIZATION': lambda x: ', '.join(sorted(set(x.dropna()))) if x.notna().any() else None,
    'MALWARE CLASS': lambda x: ', '.join(sorted(set(x.dropna()))) if x.notna().any() else None,
    'CVE_NBR': lambda x: ', '.join(sorted(set(x.dropna()))) if x.notna().any() else None
}).reset_index()

print(f"Loaded {len(threat_actor_records)} {THREAT_CATEGORY_FILTER} rows from {TAGS_FILE_PATH}")
print(f"Condensed to {len(threat_actor_condensed)} unique indicators")
display(threat_actor_condensed)

# Merge threat actor/tag context into observed_src
threat_actor_tags = threat_actor_condensed[
    ['indicator', 'tag', 'orig_tag', 'threat_category', 'NATION STATE', 'SECURITY ORGANIZATION', 'MALWARE CLASS', 'CVE_NBR']
].copy()
threat_actor_tags = threat_actor_tags.rename(columns={
    'tag': 'threat_actor',
    'orig_tag': 'threat_actor_orig_tag',
    'threat_category': 'threat_actor_category',
    'NATION STATE': 'threat_nation_state',
    'SECURITY ORGANIZATION': 'threat_security_org',
    'MALWARE CLASS': 'threat_malware_class',
    'CVE_NBR': 'threat_cve_nbr',
})
observed_src = observed_src.merge(threat_actor_tags, on='indicator', how='left')
print(f"Added threat actor context columns to observed_src")
print(f"observed_src columns after merge: {observed_src.columns.tolist()}")
display(observed_src[['indicator', 'type', 'threat_actor', 'threat_nation_state', 'threat_security_org', 'threat_cve_nbr']])


Loaded 4269 THREAT ACTOR rows from Z:\HTOC\Data_Analytics\Data\Observed_Tags\htoc_observed_indicator_tags.csv
Condensed to 1717 unique indicators


,indicator,type,orig_tag,tag,threat_category,NATION STATE,SECURITY ORGANIZATION,MALWARE CLASS,CVE_NBR
0,1-you.njalla.no,Host,"Brass Typhoon, wicked panda",Wicked Panda,THREAT ACTOR,China,MSS,None,None
1,1.20.169.90,Address,"Diamond Sleet, Famous Chollima, Hidden Cobra, ...","Diamond Sleet, Emerald Sleet, Famous Chollima,...",THREAT ACTOR,North Korea,RGB,None,None
2,1.4.195.14,Address,Mr Hamza Group,Mr Hamza Group,THREAT ACTOR,Iran,Hacktivist,None,None
3,1.54.101.176,Address,Mr Hamza Group,Mr Hamza Group,THREAT ACTOR,Iran,Hacktivist,None,None
4,1.zip,Host,UNC5112,UNC5112,THREAT ACTOR,Unknown,Cyber Criminal Organization,None,None
...,...,...,...,...,...,...,...,...,...
1712,www.thetruthaboutguns.com,Host,"Asylum Ambuscade, TA582, TA866","Asylum Ambuscade, TA582",THREAT ACTOR,Unknown,Cyber Criminal Organization,None,None
1713,www.totem.tech,Host,"Asylum Ambuscade, TA582, TA866","Asylum Ambuscade, TA582",THREAT ACTOR,Unknown,Cyber Criminal Organization,None,None
1714,www.ultrasound-guided-injections.co.uk,Host,"Asylum Ambuscade, TA582, TA866","Asylum Ambuscade, TA582",THREAT ACTOR,Unknown,Cyber Criminal Organization,None,None
1715,zerocap.com,Host,"Asylum Ambuscade, TA582, TA866","Asylum Ambuscade, TA582",THREAT ACTOR,Unknown,Cyber Criminal Organization,None,None


Added threat actor context columns to observed_src
observed_src columns after merge: ['id', 'dateAdded', 'ownerId', 'ownerName', 'webLink', 'type', 'lastModified', 'rating', 'confidence', 'threatAssessRating', 'threatAssessConfidence', 'threatAssessScore', 'threatAssessScoreObserved', 'threatAssessScoreFalsePositive', 'calScore', 'description', 'summary', 'observations', 'lastObserved', 'falsePositives', 'falsePositiveReportedByUser', 'privateFlag', 'active', 'activeLocked', 'ip', 'legacyLink', 'tags.data', 'source', 'associatedGroups.data', 'hostName', 'dnsActive', 'whoisActive', 'address', 'text', 'lastFalsePositive', 'sha256', 'md5', 'indicator', 'sources', 'threat_actor', 'threat_actor_orig_tag', 'threat_actor_category', 'threat_nation_state', 'threat_security_org', 'threat_malware_class', 'threat_cve_nbr']


,indicator,type,threat_actor,threat_nation_state,threat_security_org,threat_cve_nbr
0,65.49.1.236,Address,NaN,NaN,NaN,NaN
1,65.49.1.53,Address,NaN,NaN,NaN,NaN
2,64.62.156.199,Address,NaN,NaN,NaN,NaN
3,64.62.156.97,Address,NaN,NaN,NaN,NaN
4,65.49.1.175,Address,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...
3464,178.162.203.226,Address,NaN,NaN,NaN,NaN
3465,45.88.221.211,Address,NaN,NaN,NaN,NaN
3466,178.175.129.37,Address,NaN,NaN,NaN,NaN
3467,185.44.77.71,Address,NaN,NaN,NaN,NaN


In [111]:
import pandas as pd

# Configuration for observed indicators file and first-seen window
OBSERVED_INDICATORS_CSV_PATH = r"Z:\HTOC\Data_Analytics\Data\Observed_Indicators\htoc_observed_indicators.csv"
FIRSTSEEN_LOOKBACK_DAYS = 360

csv_path = OBSERVED_INDICATORS_CSV_PATH
df_obs = pd.read_csv(csv_path)

df_obs['firstseen_dt'] = pd.to_datetime(df_obs['firstseen_dt'], errors='coerce')

today = pd.Timestamp.today().normalize()
cutoff = today - pd.Timedelta(days=FIRSTSEEN_LOOKBACK_DAYS)

df_last14 = df_obs[
    (df_obs['firstseen_dt'] >= cutoff) &
    (df_obs['firstseen_dt'] <= today)
][['indicator', 'firstseen_dt']]

df_last14 = df_last14.rename(columns={'firstseen_dt': 'firstseen_date'})

observed_src = observed_src.merge(
    df_last14[['indicator', 'firstseen_date']],
    on='indicator',
    how='left'
)

observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,threatAssessRating,...,indicator,sources,threat_actor,threat_actor_orig_tag,threat_actor_category,threat_nation_state,threat_security_org,threat_malware_class,threat_cve_nbr,firstseen_date
0,6755399459078631,2025-06-25T17:45:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:11:02Z,3.0,46.0,3.00,...,65.49.1.236,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-25
1,6755399459078632,2025-06-25T17:45:43Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:51Z,3.0,41.0,3.00,...,65.49.1.53,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-25
2,6755399460003050,2025-06-30T12:21:10Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:39Z,3.0,40.0,3.00,...,64.62.156.199,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-30
3,6755399460003051,2025-06-30T12:21:11Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:28Z,3.0,45.0,3.00,...,64.62.156.97,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-30
4,6755399460003056,2025-06-30T12:21:18Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-03T18:10:16Z,3.0,42.0,3.00,...,65.49.1.175,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-06-30
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3464,1682885,2019-06-18T15:13:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-10T23:23:45Z,NaN,NaN,3.86,...,178.162.203.226,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025-05-09
3465,4291925,2023-01-24T03:34:26Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-05-09T12:17:54Z,4.0,0.0,3.50,...,45.88.221.211,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3466,4329148,2023-05-03T14:27:33Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-15T15:42:23Z,3.0,0.0,3.00,...,178.175.129.37,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT
3467,4501780,2024-01-18T19:06:34Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-04-13T11:09:41Z,3.0,0.0,3.00,...,185.44.77.71,HTOC Org,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT


In [112]:
# Add incidents/events as columns on observed_src

INCIDENT_COLUMN_NAME = "incidents/events"
GROUP_TYPES_OF_INTEREST = {"incident", "event"}

import re
INCIDENT_ID_REGEX = re.compile(r"\bINC\d+\b", re.IGNORECASE)

if observed_src.empty:

    observed_src[INCIDENT_COLUMN_NAME] = []

    incidents_df = pd.DataFrame()

    display("No observed data available to annotate with incidents/events.")

elif 'associatedGroups.data' not in observed_src.columns:

    observed_src[INCIDENT_COLUMN_NAME] = "None"

    incidents_df = pd.DataFrame()

    display("associatedGroups.data not present; created placeholder incidents/events column with 'None'.")

else:

    def _extract_groups_of_interest(val):

        if isinstance(val, list):

            return [item for item in val if isinstance(item, dict) and str(item.get('type', '')).lower() in GROUP_TYPES_OF_INTEREST]

        if isinstance(val, dict):

            return [val] if str(val.get('type', '')).lower() in GROUP_TYPES_OF_INTEREST else []

        return []

    def _format_group_label(item):

        if not isinstance(item, dict):

            return str(item)

        typ = str(item.get('type', '')).title()

        number = item.get('id') or item.get('xid') or item.get('name')

        return f"{typ}:{number}" if number is not None else typ

    def _extract_incidents_from_description(text):

        if not isinstance(text, str):

            return []

        matches = INCIDENT_ID_REGEX.findall(text)

        # Preserve order but remove duplicates and normalize to upper
        unique_ids = list(dict.fromkeys(m.upper() for m in matches))

        return [
            {"type": "incident", "id": inc_id}
            for inc_id in unique_ids
        ]

    groups = observed_src['associatedGroups.data'].apply(_extract_groups_of_interest)

    if 'description' in observed_src.columns:

        desc_groups = observed_src['description'].apply(_extract_incidents_from_description)

    else:

        desc_groups = pd.Series([[] for _ in range(len(observed_src))], index=observed_src.index)

    combined_groups = groups.combine(desc_groups, lambda a, b: (a or []) + (b or []))

    observed_src[INCIDENT_COLUMN_NAME] = combined_groups.apply(
        lambda lst: ";".join(_format_group_label(it) for it in lst) if isinstance(lst, list) and len(lst) > 0 else "None"
    )

    incidents_df = observed_src[combined_groups.apply(lambda lst: isinstance(lst, list) and len(lst) > 0)].copy()

    display(f"Annotated incidents/events for {len(observed_src)} indicators; found {len(incidents_df)} with matches.")

    display_cols = ['indicator', 'type', INCIDENT_COLUMN_NAME]

    display(incidents_df[display_cols])


'Annotated incidents/events for 3469 indicators; found 925 with matches.'

,indicator,type,incidents/events
38,103.120.116.162,Address,Incident:INC9385644
43,87.236.176.165,Address,Incident:INC9464932
95,3.129.187.38,Address,Incident:INC9415113
104,185.220.101.137,Address,Incident:102309
106,54.162.117.154,Address,Incident:INC9486336;Incident:INC9486332
...,...,...,...
3461,arpdcresources.ca,Host,Incident:6755399448004173;Incident:546474
3462,metalhoz.com,Host,Incident:6755399448004173;Incident:546474
3463,5.79.110.170,Address,Incident:6755399448003491;Incident:529312
3465,45.88.221.211,Address,Event:146803


In [113]:
# Unnest tags.data in observed_src to get a list of each tag per indicator

# Explode tags.data to one row per tag
tags_exploded = (
    observed_src[['indicator', 'tags.data']]
      .explode('tags.data')
      .dropna(subset=['tags.data'])
)

# Extract tag name from each tag object
tags_exploded['tag_name'] = tags_exploded['tags.data'].apply(lambda x: x.get('name') if isinstance(x, dict) else None)

# Aggregate all tag names into a list per indicator
tags_per_indicator = (
    tags_exploded.groupby('indicator')['tag_name']
    .apply(lambda x: [t for t in x if t])
    .reset_index()
    .rename(columns={'tag_name': 'tag_list'})
)

# Merge back to observed_src
observed_src_with_tags = observed_src.merge(tags_per_indicator, on='indicator', how='left')

# --- Policy/Benign tags from tags.data ---
PB_START_LOWER_TAGS = {
    "soar indicator pb",
    "scanning cdn pb",
    "known scanning pb",
    "web scanner",
}

def extract_pb_lower_tags(tag_list):
    if not isinstance(tag_list, list):
        return []
    return [
        t for t in tag_list
        if isinstance(t, str) and t.strip().lower() in PB_START_LOWER_TAGS
    ]

# Map aggregated tag_list back to observed_src
indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))

# DEBUG: Print a few values from indicator_to_tags to verify mapping
print("\nDEBUG: Sample indicator_to_tags mapping (first 5):")
for k in list(indicator_to_tags)[:5]:
    print(f"{k}: {indicator_to_tags[k]}")

# Keep full tag_list on observed_src too
observed_src['tag_list'] = observed_src['indicator'].map(lambda ind: indicator_to_tags.get(ind, []))

# DEBUG: Check a sample of tag_list for some indicators
print("\nDEBUG: Sample observed_src['tag_list'] (first 5 non-empty):")
samples = observed_src[observed_src['tag_list'].apply(lambda x: isinstance(x, list) and len(x) > 0)].head()
print(samples[['indicator', 'tag_list']])

# Store matched PB tags and a boolean flag
observed_src['pb_lower_tags'] = observed_src['indicator'].map(
    lambda ind: extract_pb_lower_tags(indicator_to_tags.get(ind, []))
)
observed_src['pb_lower_flag'] = observed_src['pb_lower_tags'].apply(
    lambda x: isinstance(x, list) and len(x) > 0
)

# DEBUG: Show how many indicators got PB tags and some examples
pb_count = observed_src['pb_lower_flag'].sum()
print(f"\nDEBUG: Number of indicators with PB tags: {pb_count}")

if pb_count > 0:
    print("\nDEBUG: Sample indicators with PB tags:")
    print(observed_src.loc[observed_src['pb_lower_flag'], ['indicator', 'tag_list', 'pb_lower_tags']].head())

# Add a human-readable reason column
observed_src['pb_lower_reason'] = observed_src['pb_lower_tags'].apply(
    lambda tags: f"pb_tag:{tags[0].strip()}" if isinstance(tags, list) and len(tags) > 0 else None
)
# DEBUG: Print a few pb_lower_reason values to check logic
print("\nDEBUG: Sample pb_lower_reason (first 5 non-null):")
print(observed_src.loc[observed_src['pb_lower_reason'].notnull(), ['indicator', 'pb_lower_reason']].head())
# Optional: show indicators with 'Scan' in tag_list
# display(observed_src_with_tags[observed_src_with_tags['tag_list'].apply(lambda tags: isinstance(tags, list) and any(isinstance(tag, str) and 'Scan' in tag for tag in tags))])

# --- Botnet tags: count and add Botnet column to observed_src ---
BOTNET_TAGS_OF_INTEREST = [
    "Scanning", "DDoS", "Spam", "Phishing", "Cryptojacking",
    "Credential Stuffing", "Ransomware", "Data Theft",
    "Cross Site Scripting Attacks", "SQL Injections",
]
tags_of_interest = BOTNET_TAGS_OF_INTEREST

def tag_counter(tag_list, tag):
    if not isinstance(tag_list, list):
        return 0
    tag_lc = tag.lower()
    return sum(1 for t in tag_list if isinstance(t, str) and t.strip().lower() == tag_lc)

tag_counts = {}
for tag in tags_of_interest:
    tag_counts[tag] = observed_src_with_tags['tag_list'].apply(lambda lst: tag_counter(lst, tag)).sum()

botnet_tags = set(tags_of_interest)
botnet_tags_lower = {b.lower() for b in botnet_tags}

def extract_botnet_tags(tag_list):
    if not isinstance(tag_list, list):
        return []
    return [t for t in tag_list if isinstance(t, str) and t.strip().lower() in botnet_tags_lower]

indicator_to_tags = dict(zip(observed_src_with_tags['indicator'], observed_src_with_tags['tag_list']))
observed_src['Botnet'] = observed_src['indicator'].map(lambda ind: extract_botnet_tags(indicator_to_tags.get(ind, [])))

pd.DataFrame(list(tag_counts.items()), columns=['Tag', 'Count'])



DEBUG: Sample indicator_to_tags mapping (first 5):
65.49.1.236: ['SOAR Indicator PB', 'DHA Splunk API', 'OS Splunk API', 'FDA Splunk API', 'CMS Splunk API', 'VA CSOC CTS Splunk', 'HRSA Splunk API', 'NIH Splunk API', 'IHS Splunk API', 'HHS Splunk API', 'Observed', 'CDC Splunk API', 'malicious']
65.49.1.53: ['SOAR Indicator PB', 'DHA Splunk API', 'OS Splunk API', 'FDA Splunk API', 'CMS Splunk API', 'VA CSOC CTS Splunk', 'HRSA Splunk API', 'NIH Splunk API', 'IHS Splunk API', 'HHS Splunk API', 'Observed', 'CDC Splunk API', 'malicious']
64.62.156.199: ['SOAR Indicator PB', 'DHA Splunk API', 'OS Splunk API', 'FDA Splunk API', 'CMS Splunk API', 'VA CSOC CTS Splunk', 'HRSA Splunk API', 'NIH Splunk API', 'IHS Splunk API', 'HHS Splunk API', 'Observed', 'CDC Splunk API', 'malicious']
64.62.156.97: ['SOAR Indicator PB', 'DHA Splunk API', 'OS Splunk API', 'FDA Splunk API', 'CMS Splunk API', 'VA CSOC CTS Splunk', 'HRSA Splunk API', 'NIH Splunk API', 'IHS Splunk API', 'HHS Splunk API', 'Observed', '

,Tag,Count
0,Scanning,8
1,DDoS,393
2,Spam,0
3,Phishing,82
4,Cryptojacking,0
5,Credential Stuffing,1
6,Ransomware,41
7,Data Theft,54
8,Cross Site Scripting Attacks,0
9,SQL Injections,0


In [114]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Configuration for OpDiv observation files
OPDIV_BASE_PATH = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#OPDIV_BASE_PATH = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
OPDIV_DATE_FORMAT = "%Y%m%d"
OPDIV_LOOKBACK_DAYS = 360

# Base file path with placeholder for date
base_path = OPDIV_BASE_PATH

def get_file_paths(base_path, days=360):
    """Generate file paths for the last `days` days using list comprehension."""
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(OPDIV_DATE_FORMAT) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        display("No files found for the specified date range.")
    else:
        display(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """Load and concatenate observed data from multiple files."""
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            display(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        display(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Fetch file paths for the last OPDIV_LOOKBACK_DAYS days
file_paths = get_file_paths(base_path, days=OPDIV_LOOKBACK_DAYS)

# Load observed data
observed_data_df = load_observed_data(file_paths)

# ── Mass Scanner Detection (Tiered) ───────────────────────────────────────────
# Tier 1: 10K–100K obs in 7 days + 5+ OpDivs — moderate raw-score multiplier (no PRISM cap).
# Tier 2: 100K+ obs in 7 days + 5+ OpDivs — heavy raw-score multiplier (no PRISM cap).
# Strong context (threat actor, incidents, TOR, etc.) can still raise the final score.
MASS_SCANNER_TIER1_OBS  = 10_000   # Tier 1 lower bound
MASS_SCANNER_TIER2_OBS  = 100_000  # Tier 2 threshold
MASS_SCANNER_OPDIV_MIN  = 5        # must hit at least 5 OpDivs (broad spread)

if not observed_data_df.empty and 'obs_date' in observed_data_df.columns:
    cutoff_7d = (pd.Timestamp.utcnow() - pd.Timedelta(days=7)).strftime('%Y-%m-%d')
    obs_7d = observed_data_df[observed_data_df['obs_date'] >= cutoff_7d].copy()

    mass_scanner_agg = (
        obs_7d.groupby('indicator')
        .agg(
            total_obs_7d    =('observations', 'sum'),
            unique_opdivs_7d=('OpDiv', 'nunique'),
            unique_days_7d  =('obs_date', 'nunique'),
        )
        .reset_index()
    )

    broad_spread = mass_scanner_agg['unique_opdivs_7d'] >= MASS_SCANNER_OPDIV_MIN

    mass_scanner_agg['mass_scanner_tier1'] = (
        (mass_scanner_agg['total_obs_7d'] >= MASS_SCANNER_TIER1_OBS) &
        (mass_scanner_agg['total_obs_7d'] <  MASS_SCANNER_TIER2_OBS) &
        broad_spread
    )
    mass_scanner_agg['mass_scanner_tier2'] = (
        (mass_scanner_agg['total_obs_7d'] >= MASS_SCANNER_TIER2_OBS) &
        broad_spread
    )

    n1 = mass_scanner_agg['mass_scanner_tier1'].sum()
    n2 = mass_scanner_agg['mass_scanner_tier2'].sum()
    display(f"Mass scanner Tier 1 (moderate penalty): {n1} indicators "
            f"({MASS_SCANNER_TIER1_OBS:,}–{MASS_SCANNER_TIER2_OBS:,} obs, {MASS_SCANNER_OPDIV_MIN}+ OpDivs)")
    display(f"Mass scanner Tier 2 (heavy penalty):  {n2} indicators "
            f"(>= {MASS_SCANNER_TIER2_OBS:,} obs, {MASS_SCANNER_OPDIV_MIN}+ OpDivs)")
    display(mass_scanner_agg[mass_scanner_agg['mass_scanner_tier2']][
        ['indicator', 'total_obs_7d', 'unique_opdivs_7d', 'unique_days_7d']
    ].sort_values('total_obs_7d', ascending=False))
else:
    mass_scanner_agg = pd.DataFrame(columns=[
        'indicator', 'total_obs_7d', 'unique_opdivs_7d', 'unique_days_7d',
        'mass_scanner_tier1', 'mass_scanner_tier2'
    ])
    display("Mass scanner detection skipped — observed_data_df is empty or missing obs_date.")



C:\Users\jaskew\AppData\Local\Temp\ipykernel_18984\518323720.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260503.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260502.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260501.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260430.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260429.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260428.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260427.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260426.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260425.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260424.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260423.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260422.csv', 'Z:/HTOC/Data_Analytics/Data/O

'Loaded data from 360 files.'

'Mass scanner Tier 1 (moderate penalty): 50 indicators (10,000–100,000 obs, 5+ OpDivs)'

'Mass scanner Tier 2 (heavy penalty):  183 indicators (>= 100,000 obs, 5+ OpDivs)'

,indicator,total_obs_7d,unique_opdivs_7d,unique_days_7d
1205,45.142.193.164,12521720,8,8
1728,88.210.63.69,11007559,6,8
349,172.234.246.109,4873727,6,7
1647,77.90.185.16,4605182,7,8
471,180.167.128.203,4143343,7,8
...,...,...,...,...
292,165.154.173.226,107293,7,8
1595,65.49.1.52,106500,6,8
1606,65.49.1.82,105185,6,8
1390,64.62.156.16,104524,8,8


In [115]:
import pandas as pd
from datetime import timedelta
import warnings

warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)

# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION & SETUP
# ═══════════════════════════════════════════════════════════════════════════════

# Time cutoffs
cutoff = pd.Timestamp.utcnow()
cutoff_naive = cutoff.tz_convert(None)

# Define known partner names (standalone tags that represent partners)
KNOWN_PARTNERS = {'DHA', 'OS', 'FDA', 'CMS', 'VA', 'HRSA', 'NIH', 'IHS', 'HHS', 'CDC'}

# ═══════════════════════════════════════════════════════════════════════════════
# HELPER FUNCTIONS
# ═══════════════════════════════════════════════════════════════════════════════

def get_all_partner_indicators_from_obs(observed_data_df, cutoff_naive):
    """Get ALL indicators that have partners from observed_data_df (no minimum threshold)."""
    if observed_data_df.empty or 'OpDiv' not in observed_data_df.columns:
        return pd.DataFrame()
    
    # Ensure obs_date is datetime
    observed_data_df['obs_date'] = pd.to_datetime(observed_data_df['obs_date'], errors='coerce')
    
    # Filter by recent dates (last 60 days to match wider time window)
    recent_obs = observed_data_df[
        observed_data_df['obs_date'] >= cutoff_naive - timedelta(days=60)
    ].copy()
    
    if recent_obs.empty:
        return pd.DataFrame()
    
    # Group by indicator and count unique OpDiv (partners)
    partner_counts = (
        recent_obs.groupby('indicator')['OpDiv']
        .agg(['nunique', lambda s: ', '.join(sorted(set(s.dropna())))]).reset_index()
        .rename(columns={'nunique': 'partner_count_obs', '<lambda_0>': 'partners_from_obs'})
    )
    
    # Keep ALL indicators with partners (no minimum threshold)
    all_partner_indicators = partner_counts[partner_counts['partner_count_obs'] >= 1].copy()
    
    return all_partner_indicators

def extract_partners_from_tags(observed_src):
    """Extract partner information from ThreatConnect tags."""
    df = observed_src.copy()

    # explode tags.data
    tags_exploded = (
        df[['indicator', 'tags.data']]
          .explode('tags.data')
          .dropna(subset=['tags.data'])
    )

    # normalize all fields in tags.data into flat columns
    tags_norm = pd.json_normalize(tags_exploded['tags.data'])
    tags_norm.columns = [f"tag_{c}" for c in tags_norm.columns]

    # Replace VA CSOC CTS Splunk with VA Splunk API in tag_name
    tags_norm['tag_name'] = tags_norm['tag_name'].str.replace('VA CSOC CTS Splunk', 'VA Splunk API', regex=False)

    # re‑attach indicator
    tags_expanded = tags_exploded.reset_index(drop=True).join(tags_norm)

    # extract partners from tags ending with " Splunk API"
    tags_expanded['partner'] = tags_expanded['tag_name'].map(
        lambda n: n[:-len(' Splunk API')] if isinstance(n, str) and n.endswith(' Splunk API') else None
    )

    # aggregate each tag_* field into a list per indicator
    tag_fields = list(tags_norm.columns)
    tag_agg = (
        tags_expanded
          .groupby('indicator', as_index=False)
          .agg({
              **{f: list for f in tag_fields},
              'partner': lambda x: [p for p in dict.fromkeys(x) if p]
          })
          .rename(columns={'partner':'partners_from_tags'})
    )

    # Also extract partners from standalone tags in tag_list (tag_name column)
    def extract_standalone_partners(tag_list):
        """Extract partner names from tag_list that match known partners"""
        if not isinstance(tag_list, list):
            return []
        # Check each tag in the list to see if it matches a known partner name
        found_partners = []
        for tag in tag_list:
            if isinstance(tag, str) and tag.strip() in KNOWN_PARTNERS:
                found_partners.append(tag.strip())
        return found_partners

    # Extract standalone partners from tag_name (which contains the tag_list)
    if 'tag_name' in tag_agg.columns:
        tag_agg['standalone_partners'] = tag_agg['tag_name'].apply(extract_standalone_partners)
        
        # Combine partners from both sources (Splunk API tags and standalone tags)
        def combine_tag_partners(row):
            """Combine partners from Splunk API tags and standalone tags"""
            partners_from_splunk = row.get('partners_from_tags', [])
            partners_standalone = row.get('standalone_partners', [])
            
            # Handle both list and string formats
            if isinstance(partners_from_splunk, str):
                partners_from_splunk = [p.strip() for p in partners_from_splunk.split(',') if p.strip()]
            if not isinstance(partners_from_splunk, list):
                partners_from_splunk = []
            
            # Combine and deduplicate
            all_partners = list(dict.fromkeys(partners_from_splunk + partners_standalone))
            return all_partners
        
        tag_agg['partners_from_tags'] = tag_agg.apply(combine_tag_partners, axis=1)
        
        # Drop the temporary standalone_partners column
        tag_agg = tag_agg.drop(columns=['standalone_partners'], errors='ignore')

    return tag_agg, tag_fields

def combine_partners_from_sources(base_agg, tag_agg, all_partner_indicators):
    """Combine partner information from both observation data and tags."""
    # Merge tag aggregation
    agg_df = base_agg.merge(tag_agg, on='indicator', how='left')
    
    # Merge with all_partner_indicators to get partners from observed_data_df
    if not all_partner_indicators.empty:
        agg_df = agg_df.merge(
            all_partner_indicators[['indicator', 'partners_from_obs', 'partner_count_obs']],
            on='indicator',
            how='left'
        )
    else:
        # Add empty columns if no partner indicators found
        agg_df['partners_from_obs'] = ''
        agg_df['partner_count_obs'] = 0
    
    # Combine partners from all sources
    def combine_all_partners(row):
        """Combine partners from observed_data_df and ThreatConnect tags."""
        obs_partners = row.get('partners_from_obs', '')
        tag_partners = row.get('partners_from_tags', [])
        
        combined = set()
        
        # Add partners from observed_data_df
        if pd.notna(obs_partners) and obs_partners:
            for p in str(obs_partners).split(', '):
                if p.strip():
                    combined.add(p.strip())
        
        # Add partners from tags
        if isinstance(tag_partners, list):
            for p in tag_partners:
                if p and p.strip():
                    combined.add(p.strip())
        elif pd.notna(tag_partners) and tag_partners:
            # Handle case where tag_partners might be a string
            for p in str(tag_partners).split(','):
                if p.strip():
                    combined.add(p.strip())
        
        return ', '.join(sorted(combined)) if combined else ''
    
    # Create combined partners column
    agg_df['partners'] = agg_df.apply(combine_all_partners, axis=1)
    
    # Calculate partner count
    agg_df['partner_count'] = agg_df['partners'].apply(
        lambda x: len([p for p in str(x).split(', ') if p.strip()]) if pd.notna(x) and x else 0
    )
    
    # Clean up temporary columns
    cols_to_drop = ['partners_from_obs', 'partner_count_obs', 'partners_from_tags']
    agg_df = agg_df.drop(columns=[col for col in cols_to_drop if col in agg_df.columns], errors='ignore')
    
    return agg_df

# ═══════════════════════════════════════════════════════════════════════════════
# MAIN PROCESSING PIPELINE
# ═══════════════════════════════════════════════════════════════════════════════

print("Starting enhanced partner extraction pipeline...")

# Step 1: Get ALL indicators with partners from observed_data_df (no filtering)
print("Identifying ALL indicators with partners from observed_data_df...")
all_partner_indicators = get_all_partner_indicators_from_obs(observed_data_df, cutoff_naive)
print(f"Found {len(all_partner_indicators)} indicators with partners from observation data")

# Step 2: Extract partners from ThreatConnect tags
print("Extracting partners from ThreatConnect tags...")
tag_agg, tag_fields = extract_partners_from_tags(observed_src)
print(f"Processed tags for {len(tag_agg)} indicators")

# Step 3: Core aggregation of other columns
print("Performing core aggregation...")
df = observed_src.copy()

first_cols = [
    'id','dateAdded','ownerId','ownerName','webLink','type','lastModified', 'falsePositives',
    'rating','confidence','description','summary','observations',
    'lastObserved','privateFlag','active','activeLocked','ip',
    'legacyLink','source','address','url', 'threatAssessScore', 'calScore', 'incidents/events', 'sources',
    'threat_actor','firstseen_date', 'tag_list','pb_lower_tags','pb_lower_flag','pb_lower_reason'
]

# Add 'Botnet' column from observed_src if it exists
if 'Botnet' in observed_src.columns:
    df['Botnet'] = observed_src['Botnet']
    first_cols.append('Botnet')

base_agg = (
    df
      .drop(columns=[
          'createdBy.id','createdBy.userName','createdBy.firstName','createdBy.lastName',
          'createdBy.pseudonym','createdBy.owner','xid','eventType','documentDateAdded',
          'documentType','fileSize','fileName','downVoteCount','upVoteCount','type_group',
          'webLink_group','ownerName_group','ownerId_group','dateAdded_group','id_group',
          'platforms.count','tactics.count',
      ], errors='ignore')
      .groupby('indicator', as_index=False)[
          [c for c in first_cols if c in df.columns]
      ]
      .first()
)

# Step 4: Combine partners from all sources
print("Combining partners from all sources...")
agg_df = combine_partners_from_sources(base_agg, tag_agg, all_partner_indicators)

# Step 5: Clean up and format list columns
print("Cleaning up list columns...")
def clean_list(lst):
    if not isinstance(lst, list):
        return []
    cleaned = []
    for v in lst:
        # drop any null‑like values
        try:
            if pd.isna(v):
                continue
        except Exception:
            pass
        # drop empty strings
        if isinstance(v, str) and v == "":
            continue
        cleaned.append(v)
    return cleaned

def list_to_csv(lst):
    """
    Takes a cleaned list and returns:
      - a comma-separated string of its items, OR
      - an empty string if there are none.
    """
    if not lst:
        return ""
    return ", ".join(str(v) for v in lst)

# Apply cleaning to tag fields (but not partners, which is already a string)
for col in ['group_ids', 'group_names'] + tag_fields:
    if col in agg_df.columns:
        agg_df[col] = agg_df[col].apply(clean_list).apply(list_to_csv)

print(f"Processing complete! Final dataset has {len(agg_df)} indicators")

# Merge mass scanner tier flags computed in Cell 8
if not mass_scanner_agg.empty:
    agg_df = agg_df.merge(
        mass_scanner_agg[[
            'indicator', 'total_obs_7d', 'unique_opdivs_7d',
            'mass_scanner_tier1', 'mass_scanner_tier2'
        ]],
        on='indicator',
        how='left'
    )
    agg_df['mass_scanner_tier1'] = agg_df['mass_scanner_tier1'].fillna(False).astype(bool)
    agg_df['mass_scanner_tier2'] = agg_df['mass_scanner_tier2'].fillna(False).astype(bool)
    agg_df['total_obs_7d']       = agg_df['total_obs_7d'].fillna(0).astype(int)
    agg_df['unique_opdivs_7d']   = agg_df['unique_opdivs_7d'].fillna(0).astype(int)
    print(f"Mass scanner flags merged — Tier 1: {agg_df['mass_scanner_tier1'].sum()} | Tier 2: {agg_df['mass_scanner_tier2'].sum()}")
else:
    agg_df['mass_scanner_tier1'] = False
    agg_df['mass_scanner_tier2'] = False
    agg_df['total_obs_7d']       = 0
    agg_df['unique_opdivs_7d']   = 0

display(agg_df)

Starting enhanced partner extraction pipeline...
Identifying ALL indicators with partners from observed_data_df...
Found 3577 indicators with partners from observation data
Extracting partners from ThreatConnect tags...
Processed tags for 3458 indicators
Performing core aggregation...
Combining partners from all sources...
Cleaning up list columns...
Processing complete! Final dataset has 3469 indicators
Mass scanner flags merged — Tier 1: 50 | Tier 2: 183


C:\Users\jaskew\AppData\Local\Temp\ipykernel_18984\778393339.py:283: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  agg_df['mass_scanner_tier1'] = agg_df['mass_scanner_tier1'].fillna(False).astype(bool)
C:\Users\jaskew\AppData\Local\Temp\ipykernel_18984\778393339.py:284: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  agg_df['mass_scanner_tier2'] = agg_df['mass_scanner_tier2'].fillna(False).astype(bool)


,indicator,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,falsePositives,rating,...,tag_name,tag_lastUsed,tag_description,tag_techniqueId,partners,partner_count,total_obs_7d,unique_opdivs_7d,mass_scanner_tier1,mass_scanner_tier2
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,142366,2016-03-21T16:46:39Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,File,2026-02-07T22:24:12Z,0,3.0,...,ransomware,2026-03-30T14:23:09Z,,,,0,0,0,False,False
1,1-you.njalla.no,5629499535004222,2025-02-11T16:19:08Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2026-04-30T07:02:29Z,0,5.0,...,"HHS-CTIR, CVE-2017-0199, Cobalt Strike, Beacon...","2025-11-13T14:00:11Z, 2025-08-11T13:35:06Z, 20...","Adversaries may buy, lease, rent, or obtain in...","T1583, T1098","NIH, VA",2,0,0,False,False
2,1.192.18.4,6755399447111259,2025-05-14T17:47:35Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-30T17:12:36Z,0,3.0,...,"SOAR Indicator PB, OS Splunk API, FDA Splunk A...","2026-04-29T18:00:10Z, 2026-05-03T17:04:42Z, 20...",Observations reported by the HHS Ofice of the ...,,"CMS, FDA, HRSA, NIH, OS",5,0,0,False,False
3,1.20.169.90,6755399493005624,2026-02-26T14:09:01Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-05-01T07:07:50Z,0,5.0,...,"VA OIT CSOC CTS Splunk, Famous Chollima, Otter...","2026-05-03T18:02:43Z, 2026-04-15T14:58:24Z, 20...",North Korean Reconnaissance General Bureau (RGB),,VA,1,0,0,False,False
4,1.234.83.26,5629499582023671,2026-01-03T16:43:23Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2026-04-29T07:17:22Z,0,3.0,...,"SOAR Indicator PB, suspicious, I&W","2026-04-29T18:00:10Z, 2026-04-28T12:46:26Z, 20...",This tag is used specifically for HTOC I&W rep...,,,0,0,0,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3464,www.sourcefiles.org,5629499559316608,2025-07-14T15:36:16Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2026-05-03T07:16:37Z,0,5.0,...,"Secret Blizzard, UNC638, Venomous Bear, VA Spl...","2025-07-14T00:00:00Z, 2025-07-14T00:00:00Z, 20...",,,VA,1,0,0,False,False
3465,www.sthda.com,4565837,2024-04-16T17:33:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2026-04-29T07:17:33Z,0,4.0,...,"DHA Splunk API, OS Splunk API, VA OIS CSOC CTS...","2026-05-03T18:00:35Z, 2026-05-03T17:04:42Z, 20...",Observations reported by the HHS Ofice of the ...,,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",8,0,0,False,False
3466,yourpensionmeeting.com,5629499559400773,2025-07-15T15:07:04Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2026-04-29T07:17:22Z,0,3.0,...,"VA OIT CSOC CTS Splunk, DHA Splunk API, CMS Sp...","2026-05-03T18:02:43Z, 2026-05-03T18:00:35Z, 20...",,,"CMS, DHA, NIH, VA",4,0,0,False,False
3467,zerogov.com,5629499559316575,2025-07-14T15:36:15Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Host,2026-05-03T07:14:40Z,0,5.0,...,"Secret Blizzard, UNC638, Venomous Bear, VA Spl...","2025-07-14T00:00:00Z, 2025-07-14T00:00:00Z, 20...",,,VA,1,0,0,False,False


In [116]:
# ──────────────────────────────────────────────────────────────────────────────
# Clean enrichment: filter unsupported types, use ID when available, parallelize
# ──────────────────────────────────────────────────────────────────────────────
import urllib.parse
import pandas as pd
from concurrent.futures import ThreadPoolExecutor, as_completed

COL_PATH = "data.enrichment.data"   # adjust if API changes

# Determine candidate indicators (use 'indicator' if present, else 'summary')
key_col = 'indicator' if 'indicator' in agg_df.columns else 'summary'

# Provider allowlists: only call API for types that support VT and/or Shodan
VT_TYPES = {'Address', 'IPv4', 'IPv6', 'Host', 'Domain', 'URL', 'File', 'SHA1', 'SHA256', 'MD5'}
SHODAN_TYPES = {'Address', 'IPv4', 'IPv6'}

# Build candidates and include 'id' when present (for ID-based lookup)
cols = [key_col, 'type']
if 'id' in agg_df.columns:
    cols.append('id')

candidates = (
    agg_df[cols]
    .dropna(subset=[key_col])
    .astype({key_col: str})
    .drop_duplicates(subset=[key_col])
)

# Pre-filter: skip types that cannot be enriched (e.g. EmailAddress)
candidates = candidates[
    candidates['type'].astype(str).str.strip().isin(VT_TYPES | SHODAN_TYPES)
].copy()

indicator_values = candidates[key_col].tolist()
display(f"Enriching {len(indicator_values)} indicators (VT; Shodan for IP types only)...")

def _enrich_one(row_series):
    """Call enrich API for one indicator. Returns (data_dict, None) on success or (None, fail_dict) on failure."""
    value = row_series[key_col]
    typ = str(row_series.get('type', '') or '')
    row_id = row_series.get('id')
    use_id = pd.notna(row_id) and str(row_id).strip().isdigit()

    try:
        # Prefer ID-based URL to avoid "More than one indicator matches" errors
        if use_id:
            indicator_id_or_value = str(int(float(row_id)))
        else:
            indicator_id_or_value = urllib.parse.quote(value, safe="")

        providers = []
        if typ in VT_TYPES:
            providers.append("VirusTotalV3")
        if typ in SHODAN_TYPES:
            providers.append("Shodan")
        if not providers:
            providers.append("VirusTotalV3")

        q = urllib.parse.urlencode({"type": providers}, doseq=True)
        enrich_url = f"/v3/indicators/{indicator_id_or_value}/enrich?{q}"

        ro = RequestObject()
        ro.set_http_method("POST")
        ro.set_request_uri(enrich_url)
        ro.set_body({})
        resp = tc.api_request(ro)

        try:
            data = resp.json()
        except Exception:
            data = {"status": getattr(resp, 'status_code', 'n/a'), "raw": getattr(resp, 'text', None)}

        data[key_col] = value
        return (data, None)
    except Exception as e:
        return (None, {key_col: value, "type": typ, "error": str(e)})

enriched_results = []
failed = []
max_workers = 8

with ThreadPoolExecutor(max_workers=max_workers) as executor:
    futures = {executor.submit(_enrich_one, row): row for _, row in candidates.iterrows()}
    for future in as_completed(futures):
        result, err = future.result()
        if result is not None:
            enriched_results.append(result)
        else:
            failed.append(err)

# If nothing enriched, carry on with base
if not enriched_results:
    display("No enrichment data retrieved.")
    recent_tags = agg_df.copy()
else:
    df_enriched = (
        pd.json_normalize(enriched_results)
        .drop_duplicates(subset=[key_col], keep="last")
    )

    # Merge enrichment block into base
    recent_tags = agg_df.merge(df_enriched, on=key_col, how="left")

    # ── Flatten enrichment payload without creating duplicate base rows ───────
    if COL_PATH in recent_tags.columns:
        exploded = (
            recent_tags[[key_col, COL_PATH]]
            .explode(COL_PATH)
            .dropna(subset=[COL_PATH])
        )

        enrich_flat = pd.json_normalize(exploded[COL_PATH]).add_prefix("enrich_")
        enrich_flat[key_col] = exploded[key_col].values

        def _flatten_lists(values):
            out = []
            for v in values:
                if isinstance(v, list):
                    out.extend(v)
                else:
                    out.append(v)
            return out

        def _agg_obj(series: pd.Series):
            vals = [v for v in series.dropna()]
            if not vals:
                return None
            flat = _flatten_lists(vals)
            if all(not isinstance(v, (list, dict)) for v in flat):
                arr = pd.Series(flat).unique()
                return list(arr)
            return flat

        obj_cols = enrich_flat.select_dtypes("object").columns.difference([key_col])
        num_cols = enrich_flat.columns.difference(obj_cols.union({key_col}))

        agg_dict = {c: _agg_obj for c in obj_cols}
        agg_dict.update({c: "max" for c in num_cols})

        enrich_wide = (
            enrich_flat
            .groupby(key_col, as_index=False)
            .agg(agg_dict)
        )

        recent_tags = (
            recent_tags.drop(columns=[COL_PATH], errors="ignore")
                       .drop_duplicates(subset=[key_col])
                       .merge(enrich_wide, on=key_col, how="left")
        )

    display(f"Enrichment complete for {recent_tags[key_col].notna().sum()} indicators.")

# Compact failure summary without flooding output
if failed:
    fail_df = pd.DataFrame(failed)
    display(f"{len(failed)} indicators failed enrichment (showing up to 10):")
    display(fail_df.head(10))

# Show preview of key enrichment columns if present
preview_cols = [c for c in [key_col, 'enrich_hostNames', 'enrich_domains', 'enrich_tags', 'enrich_vtMaliciousCount'] if c in recent_tags.columns]

recent_tags.drop(
    columns=[
        'tag_id', 'tag_lastUsed', 'tag_lastModified', 'tag_ownerId',
        'tag_ownerName', 'tag_dateAdded', 'tag_description','tag_tactics.count',
        'tag_platform.data', 'tag_platform.count', 'data.id', 'data.dateAdded', 'data.ownerId',
        'data.webLink', 'data.ownerName', 'data.lastModified', 'data.summary', 'data.ip',
        'data.legacyLink','data.source', 'enrich_cloudProvider', 'enrich_cloudRegion', 'enrich_type',
        'id'
    ],
    inplace=True,
    errors='ignore'
)

display(recent_tags[preview_cols])

'Enriching 3424 indicators (VT; Shodan for IP types only)...'

Status Code: 400
Failed API Response: b'{"message":"Unable to enrich this indicator. Either the indicator or enrichment types are not compatible, or some other kind of error occurred while trying to get the enrichment information.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"message":"Unable to enrich this indicator. Either the indicator or enrichment types are not compatible, or some other kind of error occurred while trying to get the enrichment information.","status":"Error"}'


'Enrichment complete for 3469 indicators.'

'2 indicators failed enrichment (showing up to 10):'

,indicator,type,error
0,autorun.inf,Host,"b'{""message"":""Unable to enrich this indicator...."
1,system.net.webclient,Host,"b'{""message"":""Unable to enrich this indicator...."


,indicator,enrich_hostNames,enrich_domains,enrich_tags,enrich_vtMaliciousCount
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,None,None,None,NaN
1,1-you.njalla.no,None,None,None,NaN
2,1.192.18.4,None,None,None,6.0
3,1.20.169.90,None,None,None,0.0
4,1.234.83.26,None,None,None,3.0
...,...,...,...,...,...
3464,www.sourcefiles.org,None,None,None,NaN
3465,www.sthda.com,None,None,None,NaN
3466,yourpensionmeeting.com,None,None,None,NaN
3467,zerogov.com,None,None,None,NaN


In [117]:
# Count how many indicators are associated with a unique enrich_domains value

# Only proceed if 'enrich_domains' exists in exploded DataFrame
if 'enrich_domains' in exploded.columns:
    # Drop rows where enrich_domains is missing or NaN
    domains_df = exploded.dropna(subset=['enrich_domains'])

    # If enrich_domains is a list, flatten to individual domain rows
    def flatten_domains(row):
        val = row['enrich_domains']
        if isinstance(val, list):
            return [(row['indicator'], d) for d in val if pd.notna(d)]
        elif pd.notna(val):
            return [(row['indicator'], val)]
        return []

    flat = []
    for _, row in domains_df.iterrows():
        flat.extend(flatten_domains(row))

    flat_df = pd.DataFrame(flat, columns=['indicator', 'domain'])

    # Count unique indicators per domain
    domain_counts = (
        flat_df.groupby('domain')['indicator']
        .nunique()
        .reset_index()
        .rename(columns={'indicator': 'indicator_count'})
        .sort_values('indicator_count', ascending=False)
    )

    display(domain_counts)
else:
    display("Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count.")

"Column 'enrich_domains' does not exist in the provided DataFrame. Skipping domain association count."

In [118]:
# Re-use get_file_paths/load_observed_data to pull a longer window (365 days), then add obs_count to recent_tags
OPDIV_OBS_COUNT_DAYS = 365
file_paths = get_file_paths(base_path, days=OPDIV_OBS_COUNT_DAYS)
observed_data_df = load_observed_data(file_paths)

# Aggregate by indicator (unique obs_date count) and merge into recent_tags
agg_by_indicator = (
    observed_data_df
    .groupby('indicator', as_index=False)['obs_date']
    .nunique()
    .rename(columns={'obs_date': 'obs_days_count'})
)
agg_by_indicator = agg_by_indicator[agg_by_indicator['indicator'].isin(recent_tags['indicator'])]
recent_tags = recent_tags.merge(
    agg_by_indicator.rename(columns={'obs_days_count': 'obs_count'}),
    on='indicator',
    how='left'
)
display(agg_by_indicator)


C:\Users\jaskew\AppData\Local\Temp\ipykernel_18984\518323720.py:16: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


"Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260503.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260502.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260501.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260430.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260429.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260428.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260427.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260426.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260425.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260424.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260423.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20260422.csv', 'Z:/HTOC/Data_Analytics/Data/O

'Loaded data from 365 files.'

,indicator,obs_days_count
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,2
1,1-you.njalla.no,37
7,1.192.18.4,19
8,1.20.169.90,3
10,1.234.83.26,43
...,...,...
7957,www.sourcefiles.org,1
7958,www.sthda.com,194
7968,yourpensionmeeting.com,222
7969,zerogov.com,1


In [119]:
import pandas as pd
import numpy as np
import re

# -------------------------------------------------------------------
# 0. Load your data
# -------------------------------------------------------------------
df_scored = recent_tags.copy()

# -------------------------------------------------------------------
# 0.5 Missing-value modification
# -------------------------------------------------------------------
VT_COL = 'enrich_vtMaliciousCount'
VT_EFFECTIVE_MAX = 40

if VT_COL in df_scored.columns:
    df_scored[VT_COL] = pd.to_numeric(df_scored[VT_COL], errors='coerce')
    df_scored['vt_present'] = df_scored[VT_COL].notna()
else:
    df_scored[VT_COL] = np.nan
    df_scored['vt_present'] = False

df_scored['vt_present'] = df_scored['vt_present'].astype(bool)
df_scored['vt_display'] = np.where(df_scored['vt_present'], df_scored[VT_COL], 'No VT Score')
df_scored['vt_numeric_for_scoring'] = df_scored[VT_COL].fillna(0).clip(0, VT_EFFECTIVE_MAX)

# -------------------------------------------------------------------
# 1. Rule-based Threat Scoring
# -------------------------------------------------------------------
scoring_bins = [0, 200, 500, 800, 1001]
label_bins = ['low', 'medium', 'high', 'critical']

Weights = {
    "MALICIOUS_WEIGHT": 7.50,
    "OBSERVATION_COUNT_WEIGHT": 0.02,
    "CONTINUITY_WEIGHT": 0.90,
    "TC_RATING": 0.01,
    "TC_CONFIDENCE": 0.025,
    "TOR_ACTIVITY_WEIGHT": 9.00,
    "CAL_SCORE_WEIGHT": 2.75,
    "TC_THREAT_SCORE_WEIGHT": 0.75,
    "INCIDENTS_EVENTS_WEIGHT": 8.00,
    "PARTNER_WEIGHT": 2.10,
    "SOURCES_WEIGHT": 2.80,
    "THREAT_ACTOR_WEIGHT": 10.00,
    "FIRST_OBS_WEIGHT": 2.00
}

BOTNET_ACTIONS = {
    'scanning', 'ddos', 'spam', 'phishing', 'cryptojacking',
    'credential stuffing', 'ransomware'
}

TOR_ACTIVITY = {'tor', 'tor activity'}

PB_START_LOWER_TAGS = {
    'soar indicator pb',
    'scanning cdn pb',
    'known scanning pb',
    'web scanner',
}

MAX_OBS_REALISTIC = 365
MAX_RATING = 5
MAX_CONFIDENCE = 100

FALSE_POSITIVE_WEIGHT = 0.9
BOTNET_MULTIPLIER = 0.4
SCANNER_PENALTY_MULTIPLIER = 0.80
DATA_QUALITY_FLOOR = 0.85
PB_BASE_SCORE_MULTIPLIER = 0.45
THREAT_TAG_MIN_FLOOR = 560

# -------------------------------------------------------------------
# Utility Functions
# -------------------------------------------------------------------
def convert_to_list(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    if isinstance(val, (list, set, tuple)):
        return list(val)
    if isinstance(val, str):
        s = val.strip()
        if s.startswith('[') and s.endswith(']'):
            try:
                import ast
                parsed = ast.literal_eval(s)
                if isinstance(parsed, (list, tuple)):
                    return list(parsed)
            except Exception:
                pass
        return [x.strip() for x in val.split(',') if x.strip()]
    return [val] if val else []

def extract_pb_lower_tags_from_val(val):
    vals = convert_to_list(val)
    return [
        str(x).strip()
        for x in vals
        if str(x).strip().lower() in PB_START_LOWER_TAGS
    ]

def has_pb_lower_tag(val):
    return len(extract_pb_lower_tags_from_val(val)) > 0

def normalize_token(val):
    s = str(val).strip().lower()
    s = s.replace('_', ' ')
    s = re.sub(r'[()]', lambda m: f" {m.group(0)} ", s)
    s = re.sub(r'\s*/\s*', ' / ', s)
    s = " ".join(s.split())
    return s

def flatten_tag_tokens(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return []
    values = val if isinstance(val, (list, set, tuple)) else [val]
    out = []
    for item in values:
        if item is None or (isinstance(item, float) and pd.isna(item)):
            continue
        if isinstance(item, dict):
            tag_name = item.get('name')
            if tag_name is not None:
                out.append(normalize_token(tag_name))
            continue
        if isinstance(item, str):
            s = item.strip()
            if not s:
                continue
            parsed = convert_to_list(s)
            if isinstance(parsed, list) and len(parsed) > 1:
                out.extend(normalize_token(x) for x in parsed if str(x).strip())
            else:
                out.extend(normalize_token(x) for x in s.split(',') if str(x).strip())
            continue
        out.append(normalize_token(item))
    return [t for t in out if t not in {'', 'none', 'nan'}]

def token_matches_pattern(token, pattern):
    p = normalize_token(pattern)
    if p.startswith('*'):
        return token.endswith(p[1:])
    return token == p

def has_threat_actor(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    if isinstance(val, str):
        s = val.strip()
        return not (s == '' or s.lower() in {'none', 'nan'})
    if isinstance(val, (list, set, tuple)):
        return len(val) > 0
    return False

# -------------------------------------------------------------------
# Input caps
# -------------------------------------------------------------------
df_scored['obs_count'] = pd.to_numeric(
    df_scored['obs_count'] if 'obs_count' in df_scored.columns else pd.Series(0, index=df_scored.index),
    errors='coerce'
).fillna(0).clip(0, MAX_OBS_REALISTIC)

df_scored['rating'] = pd.to_numeric(
    df_scored['rating'] if 'rating' in df_scored.columns else pd.Series(0, index=df_scored.index),
    errors='coerce'
).fillna(0).clip(0, MAX_RATING)

df_scored['confidence'] = pd.to_numeric(
    df_scored['confidence'] if 'confidence' in df_scored.columns else pd.Series(0, index=df_scored.index),
    errors='coerce'
).fillna(0).clip(0, MAX_CONFIDENCE)

df_scored['calScore'] = pd.to_numeric(
    df_scored['calScore'] if 'calScore' in df_scored.columns else pd.Series(0, index=df_scored.index),
    errors='coerce'
).fillna(0).clip(0, 1000)

df_scored['type'] = (
    df_scored['type'] if 'type' in df_scored.columns else pd.Series('', index=df_scored.index)
).astype(str)

# -------------------------------------------------------------------
# PB lower-start flag/reason
# -------------------------------------------------------------------
if 'pb_lower_flag' in df_scored.columns:
    df_scored['pb_lower_flag'] = df_scored['pb_lower_flag'].fillna(False).astype(bool)
elif 'tag_list' in df_scored.columns:
    df_scored['pb_lower_flag'] = df_scored['tag_list'].apply(has_pb_lower_tag).astype(bool)
else:
    df_scored['pb_lower_flag'] = False

if 'pb_lower_tags' not in df_scored.columns:
    if 'tag_list' in df_scored.columns:
        df_scored['pb_lower_tags'] = df_scored['tag_list'].apply(extract_pb_lower_tags_from_val)
    else:
        df_scored['pb_lower_tags'] = [[] for _ in range(len(df_scored))]

if 'pb_lower_reason' not in df_scored.columns:
    df_scored['pb_lower_reason'] = df_scored['pb_lower_tags'].apply(
        lambda tags: f"pb_tag:{str(tags[0]).strip()}" if isinstance(tags, list) and len(tags) > 0 else None
    )

# -------------------------------------------------------------------
# First-seen recency boost
# -------------------------------------------------------------------
FIRST_OBS_MAX_DAYS = 14
firstseen_col = df_scored.get('firstseen_date', pd.Series(pd.NaT, index=df_scored.index))
firstseen_dt = pd.to_datetime(firstseen_col, errors='coerce')
today = pd.Timestamp.today().normalize()
age_days = (today - firstseen_dt).dt.days.clip(lower=0)
freshness = ((FIRST_OBS_MAX_DAYS - age_days) / FIRST_OBS_MAX_DAYS).clip(lower=0.0, upper=1.0)
freshness = freshness.where(firstseen_dt.notna(), 0.0)
df_scored['first_obs_raw_score'] = freshness * Weights['FIRST_OBS_WEIGHT']

# -------------------------------------------------------------------
# Base additive evidence
# -------------------------------------------------------------------
MALICIOUS_EXPONENT = 0.75
df_scored['w_malicious_eff'] = Weights['MALICIOUS_WEIGHT']
df_scored['w_tc_rating_eff'] = Weights['TC_RATING']

df_scored['malicious_scaled'] = np.power(df_scored['vt_numeric_for_scoring'], MALICIOUS_EXPONENT)
df_scored['malicious_raw_score'] = df_scored['malicious_scaled'] * Weights['MALICIOUS_WEIGHT']

FILE_TYPES = {'SHA1', 'SHA256', 'MD5', 'file', 'File'}
df_scored['is_file_type'] = df_scored['type'].isin(FILE_TYPES)

df_scored['continuity_val'] = df_scored['type'].map({
    'Address': 1, 'IPv4': 1, 'IPv6': 1,
    'Domain': 2, 'Host': 2, 'URL': 2, 'Stripped URL': 2,
    'EmailAddress': 2, 'EmailSubject': 2,
    'SHA1': 3, 'SHA256': 3, 'MD5': 3, 'file': 3, 'File': 3
}).fillna(0)

df_scored['continuity_raw_score'] = df_scored['continuity_val'] * Weights['CONTINUITY_WEIGHT']
df_scored.loc[df_scored['is_file_type'], 'continuity_raw_score'] = 900

df_scored['tc_raw_rating'] = df_scored['rating'] * df_scored['w_tc_rating_eff']
df_scored['tc_raw_confidence'] = np.sqrt(df_scored['confidence']) * Weights['TC_CONFIDENCE']
df_scored['cal_raw_score'] = (df_scored['calScore'] / 1000.0) * Weights['CAL_SCORE_WEIGHT']

TC_THREAT_COL = 'threatAssessScore'
if TC_THREAT_COL in df_scored.columns:
    df_scored[TC_THREAT_COL] = pd.to_numeric(df_scored.get(TC_THREAT_COL, 0), errors='coerce').fillna(0).clip(0, 1000)
    df_scored['tc_threat_raw_score'] = (df_scored[TC_THREAT_COL] / 1000.0) * Weights['TC_THREAT_SCORE_WEIGHT']
else:
    df_scored['tc_threat_raw_score'] = 0.0

# -------------------------------------------------------------------
# Incident/Event association bonus
# PB-lowered indicators do NOT get this boost.
# -------------------------------------------------------------------
INCIDENTS_EVENTS_COL = 'incidents/events'

def has_incident_event(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    if isinstance(val, (list, set, tuple)):
        return len(val) > 0
    s = str(val).strip()
    return not (s == '' or s.lower() in {'none', 'nan'})

if INCIDENTS_EVENTS_COL in df_scored.columns:
    df_scored['incidents_events_flag'] = df_scored[INCIDENTS_EVENTS_COL].apply(has_incident_event).astype(int)
else:
    df_scored['incidents_events_flag'] = 0

df_scored['incidents_events_score'] = np.where(
    df_scored['pb_lower_flag'],
    0.0,
    df_scored['incidents_events_flag'] * Weights['INCIDENTS_EVENTS_WEIGHT']
)

# -------------------------------------------------------------------
# Sources / Partners
# -------------------------------------------------------------------
def count_sources(sources_val):
    if sources_val is None or (isinstance(sources_val, float) and pd.isna(sources_val)):
        return 0
    if isinstance(sources_val, str):
        return len(set(s.strip() for s in sources_val.split(',') if s.strip()))
    if isinstance(sources_val, (list, set, tuple)):
        return len(set(str(s).strip() for s in sources_val if str(s).strip()))
    return 0

df_scored['sources_count'] = df_scored['sources'].apply(count_sources) if 'sources' in df_scored.columns else 1
df_scored['sources_count_safe'] = df_scored['sources_count'].clip(lower=1)
df_scored['sources_raw_score'] = np.log1p(df_scored['sources_count_safe'] - 1) * Weights['SOURCES_WEIGHT']

def count_partners(partners_val):
    if partners_val is None or (isinstance(partners_val, float) and pd.isna(partners_val)):
        return 0
    if isinstance(partners_val, (list, set, tuple)):
        return len(set(str(x).strip() for x in partners_val if str(x).strip()))
    s = str(partners_val).strip()
    if s == '' or s.lower() in {'none', 'nan'}:
        return 0
    return len(set(str(x).strip() for x in convert_to_list(s) if str(x).strip()))

df_scored['partners_count'] = df_scored['partners'].apply(count_partners) if 'partners' in df_scored.columns else 0
df_scored['partners_count_safe'] = df_scored['partners_count'].clip(lower=1)
df_scored['partner_raw_score'] = np.log1p(df_scored['partners_count_safe'] - 1) * Weights['PARTNER_WEIGHT']

# -------------------------------------------------------------------
# Tag boost logic
# -------------------------------------------------------------------
PAIRED_COUNTRY_TAGS = {
    'iran': ['MOIS', 'IRGC', 'IRGC CEC', 'IRGC EWCD', 'IRGC QF', '*Sandstorm', '*Kitten'],
    'russia': ['SVR', 'FSB', 'GRU', '*Blizzard', '*Bear'],
    'china': ['MSS', 'ShSSB', 'TSSB', 'GSSB', 'JSSB', 'SSSB', 'HuSSB', 'HaSSB', 'SiSSB', 'PLA', 'PLA CSF', 'PLA SSF', 'PLAN', '*Panda', '*Typhoon'],
    'north korea': ['RGB', '*Chollima', '*Sleet'],
    'palestine': ['Hamas'],
    'lebanon': ['Lebanese Hizballah'],
}

STANDALONE_BOOST_TAGS = {
    'vietnam', 'belarus', 'palestine', 'pakistan', 'india',
    'teampcp', 'compromised trivy', 'operational relay box',
    'command and control', 'command and control (c2)', 'c2',
    'data exfiltration',
    'wiper', 'destructive wiper', 'data wiper',
    'dropper', 'loader', 'loader/dropper', 'loader / dropper',
    'backdoor', 'rat', 'backdoor/rat', 'backdoor / rat',
    'ransomware',
    'remote code execution', 'remote code execution (rce)', 'rce',
    'cisco', 'fortigate', 'fortinet', 'sap netweaver',
    'apt & targeted attack', 'spacehop orb', 'superjumper',
}

CVE_PATTERN = re.compile(r'^cve-\d{1,4}-\d{1,7}$', re.IGNORECASE)

def evaluate_tagging_boost_reason(row):
    token_fields = [
        'adversary', 'threat_actor', 'threat_actor_orig_tag', 'threat_actor_category',
        'threat_nation_state', 'threat_security_org', 'threat_malware_class', 'threat_cve_nbr',
        'tag_name', 'enrich_tags', 'tags.data'
    ]
    tokens = []
    for col in token_fields:
        if col in row.index:
            tokens.extend(flatten_tag_tokens(row.get(col)))

    token_set = set(tokens)

    for country, pair_tags in PAIRED_COUNTRY_TAGS.items():
        pair_hits = sorted({
            tok for tok in token_set
            for patt in pair_tags
            if token_matches_pattern(tok, patt)
        })
        if country in token_set and pair_hits:
            return f"pair:{country}+{pair_hits[0]}"

    standalone_hits = sorted(tok for tok in token_set if tok in STANDALONE_BOOST_TAGS)
    if standalone_hits:
        return f"standalone:{standalone_hits[0]}"

    cve_hits = sorted(tok for tok in token_set if CVE_PATTERN.match(tok))
    if cve_hits:
        return f"cve:{cve_hits[0]}"

    return None

threat_actor_flag = pd.Series(False, index=df_scored.index)
if 'adversary' in df_scored.columns:
    threat_actor_flag = df_scored['adversary'].apply(has_threat_actor)
elif 'threat_actor' in df_scored.columns:
    threat_actor_flag = df_scored['threat_actor'].apply(has_threat_actor)

df_scored['threat_actor_score'] = threat_actor_flag.astype(int) * Weights['THREAT_ACTOR_WEIGHT']
df_scored['Tagging_Boost_Reason'] = df_scored.apply(evaluate_tagging_boost_reason, axis=1)
df_scored['Tagging_Boost'] = df_scored['Tagging_Boost_Reason'].notna().astype(bool)

# -------------------------------------------------------------------
# TOR Activity
# -------------------------------------------------------------------
def has_tor_activity(val):
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    if isinstance(val, (list, set, tuple)):
        t = " ".join(map(str, val)).lower()
    elif isinstance(val, str):
        if not val.strip():
            return False
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
        if t in ['nan', 'none', '']:
            return False
    return any(keyword in t for keyword in TOR_ACTIVITY)

tor_mask_tag = pd.Series(False, index=df_scored.index)
tor_mask_enrich_tags = pd.Series(False, index=df_scored.index)

if 'enrich_tags' in df_scored.columns:
    tor_mask_enrich_tags = df_scored['enrich_tags'].apply(has_tor_activity)

if 'tag_name' in df_scored.columns:
    tor_mask_tag = df_scored['tag_name'].apply(convert_to_list).apply(has_tor_activity)

tor_flag = (tor_mask_enrich_tags | tor_mask_tag).astype(int)
df_scored['tor_activity_score'] = tor_flag * Weights['TOR_ACTIVITY_WEIGHT']

vt_series_present = pd.to_numeric(df_scored['vt_numeric_for_scoring'], errors='coerce').fillna(0)
boost_mask = df_scored['vt_present'] & (vt_series_present >= 10) & tor_flag.astype(bool)
df_scored.loc[boost_mask, 'tor_activity_score'] *= 2

# -------------------------------------------------------------------
# Stacked context bonus
# -------------------------------------------------------------------
df_scored['stacked_context_count'] = (
    (df_scored['threat_actor_score'] > 0).astype(int) +
    (df_scored['tor_activity_score'] > 0).astype(int) +
    (df_scored['incidents_events_score'] > 0).astype(int) +
    (df_scored['sources_count'] >= 2).astype(int) +
    (df_scored['partners_count'] >= 2).astype(int)
)

df_scored['stacked_context_bonus'] = np.select(
    [
        df_scored['stacked_context_count'] >= 4,
        df_scored['stacked_context_count'] == 3,
        df_scored['stacked_context_count'] == 2,
    ],
    [25.0, 15.0, 7.0],
    default=0.0
)

# -------------------------------------------------------------------
# Raw score
# -------------------------------------------------------------------
threat_boost_mask = df_scored['Tagging_Boost'].astype(bool)

df_scored.loc[
    threat_boost_mask,
    ['malicious_raw_score', 'tc_raw_rating', 'tc_raw_confidence', 'tc_threat_raw_score']
] = 0.0

df_scored['raw_score'] = (
    df_scored['malicious_raw_score'] +
    df_scored['continuity_raw_score'] +
    df_scored['tc_raw_rating'] +
    df_scored['tc_raw_confidence'] +
    df_scored['tor_activity_score'] +
    df_scored['incidents_events_score'] +
    df_scored['sources_raw_score'] +
    df_scored['partner_raw_score'] +
    df_scored['threat_actor_score'] +
    df_scored['cal_raw_score'] +
    df_scored['tc_threat_raw_score'] +
    df_scored['first_obs_raw_score'] +
    df_scored['stacked_context_bonus']
)

df_scored['pb_base_multiplier'] = np.where(
    df_scored['pb_lower_flag'],
    PB_BASE_SCORE_MULTIPLIER,
    1.0
)
df_scored['raw_score'] *= df_scored['pb_base_multiplier']

# -------------------------------------------------------------------
# Penalties / multipliers
# -------------------------------------------------------------------
OBS_PENALTY_STRENGTH = float(Weights['OBSERVATION_COUNT_WEIGHT'])
OBS_MIN_MULTIPLIER = 0.50

obs_frac = df_scored['obs_count'] / MAX_OBS_REALISTIC
df_scored['obs_penalty_multiplier'] = (
    1.0 - OBS_PENALTY_STRENGTH * obs_frac
).clip(OBS_MIN_MULTIPLIER, 1.0)
df_scored['raw_score'] *= df_scored['obs_penalty_multiplier']

needed = ['type', 'rating', 'confidence']
present_frac = df_scored[needed].notna().sum(axis=1) / len(needed)
df_scored['data_quality_multiplier'] = present_frac.clip(DATA_QUALITY_FLOOR, 1.0)
df_scored['raw_score'] *= df_scored['data_quality_multiplier']

col = df_scored.get('Botnet', None)
if col is None:
    df_scored['botnet_flag'] = 0
else:
    def is_botnet(val):
        text = " ".join(map(str, val)).lower() if isinstance(val, (list, set, tuple)) else str(val).lower()
        return int(any(action in text for action in BOTNET_ACTIONS))
    df_scored['botnet_flag'] = pd.Series(col).apply(is_botnet).astype(int)

botnet_penalty_mask = (
    (df_scored['botnet_flag'] == 1) &
    (~df_scored['Tagging_Boost']) &
    (~df_scored['is_file_type'])
)
df_scored['botnet_penalty_multiplier'] = 1.0
df_scored.loc[botnet_penalty_mask, 'botnet_penalty_multiplier'] = BOTNET_MULTIPLIER
df_scored['raw_score'] *= df_scored['botnet_penalty_multiplier']

if 'falsePositives' in df_scored.columns:
    df_scored['falsePositives'] = pd.to_numeric(df_scored['falsePositives'], errors='coerce').fillna(0)
    mask_fp = df_scored['falsePositives'] > 0
    df_scored['false_positive_raw_score'] = df_scored['raw_score'] * FALSE_POSITIVE_WEIGHT
    df_scored.loc[mask_fp, 'raw_score'] = df_scored.loc[mask_fp, 'false_positive_raw_score']
else:
    df_scored['falsePositives'] = 0
    df_scored['false_positive_raw_score'] = df_scored['raw_score']

def has_scanner_tag(val):
    scanners = {
        'scanner', 'masscan', 'zmap', 'shodan', 'censys',
        'active scanning: scanning ip blocks', 'web scanner', 'active scanning'
    }
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return False
    if isinstance(val, (list, set, tuple)):
        t = " ".join(map(str, val)).lower()
    elif isinstance(val, str):
        if not val.strip():
            return False
        t = " ".join(x.strip() for x in val.split(',')).lower()
    else:
        t = str(val).lower()
        if t in ['nan', 'none', '']:
            return False
    return any(s in t for s in scanners)

scanner_mask_enrich = pd.Series(False, index=df_scored.index)
scanner_mask_tag = pd.Series(False, index=df_scored.index)

if 'enrich_tags' in df_scored.columns:
    scanner_mask_enrich = df_scored['enrich_tags'].apply(has_scanner_tag)

if 'tag_name' in df_scored.columns:
    scanner_mask_tag = df_scored['tag_name'].apply(convert_to_list).apply(has_scanner_tag)

scanner_mask = (scanner_mask_enrich | scanner_mask_tag) & ~df_scored['is_file_type']
df_scored['scanner_penalty_multiplier'] = np.where(scanner_mask, SCANNER_PENALTY_MULTIPLIER, 1.0)
df_scored['raw_score'] *= df_scored['scanner_penalty_multiplier']

MASS_SCANNER_TIER1_MULTIPLIER = 0.40
MASS_SCANNER_TIER2_MULTIPLIER = 0.05

tier1_col = df_scored.get('mass_scanner_tier1', None)
tier2_col = df_scored.get('mass_scanner_tier2', None)

mass_scanner_tier1_mask = (
    tier1_col.astype(bool) & ~df_scored['is_file_type']
    if tier1_col is not None else pd.Series(False, index=df_scored.index)
)
mass_scanner_tier2_mask = (
    tier2_col.astype(bool) & ~df_scored['is_file_type']
    if tier2_col is not None else pd.Series(False, index=df_scored.index)
)

df_scored['mass_scanner_penalty_multiplier'] = np.select(
    [mass_scanner_tier2_mask, mass_scanner_tier1_mask],
    [MASS_SCANNER_TIER2_MULTIPLIER, MASS_SCANNER_TIER1_MULTIPLIER],
    default=1.0
)
df_scored['raw_score'] *= df_scored['mass_scanner_penalty_multiplier']

# -------------------------------------------------------------------
# Normalize
# -------------------------------------------------------------------
MAX_SOURCES_REALISTIC = 8
MAX_PARTNERS_REALISTIC = 10
MAX_STACKED_CONTEXT_BONUS = 25.0

BASE_CAP = (
    np.power(VT_EFFECTIVE_MAX, MALICIOUS_EXPONENT) * Weights['MALICIOUS_WEIGHT'] +
    3 * Weights['CONTINUITY_WEIGHT'] +
    (MAX_RATING * Weights['TC_RATING']) +
    (np.sqrt(MAX_CONFIDENCE) * Weights['TC_CONFIDENCE']) +
    (Weights['TOR_ACTIVITY_WEIGHT'] * 2) +
    Weights['INCIDENTS_EVENTS_WEIGHT'] +
    (np.log1p(MAX_SOURCES_REALISTIC - 1) * Weights['SOURCES_WEIGHT']) +
    (np.log1p(MAX_PARTNERS_REALISTIC - 1) * Weights['PARTNER_WEIGHT']) +
    Weights['THREAT_ACTOR_WEIGHT'] +
    Weights['CAL_SCORE_WEIGHT'] +
    Weights['TC_THREAT_SCORE_WEIGHT'] +
    Weights['FIRST_OBS_WEIGHT'] +
    MAX_STACKED_CONTEXT_BONUS
)

FILE_BASELINE_RAW = 900.0
df_scored['raw_score_cap_row'] = np.where(
    df_scored['is_file_type'],
    BASE_CAP + FILE_BASELINE_RAW,
    BASE_CAP
)

df_scored['PRISM_Score'] = (
    1000 * (df_scored['raw_score'] / df_scored['raw_score_cap_row']).clip(0, 1)
)

df_scored['PRISM_Score'] = np.minimum(
    df_scored['PRISM_Score'] * 1.40, 1000
).round().fillna(0).astype(int)

# -------------------------------------------------------------------
# VT-driven ceilings / floors
# PB-lowered indicators bypass the VT-driven HIGH floor.
# -------------------------------------------------------------------
vt_present_mask = df_scored['vt_present']
vt_counts_present = df_scored['vt_numeric_for_scoring']

low_cap_mask = vt_present_mask & (vt_counts_present <= 3)
high_floor_mask = vt_present_mask & (vt_counts_present >= 13)

tor_present_mask = df_scored['tor_activity_score'] > 0
threat_actor_present_mask = df_scored['threat_actor_score'] > 0

low_cap_final_mask = low_cap_mask & ~(tor_present_mask | threat_actor_present_mask | threat_boost_mask)
df_scored.loc[low_cap_final_mask, 'PRISM_Score'] = (
    df_scored.loc[low_cap_final_mask, 'PRISM_Score'].clip(upper=499)
)

high_floor_final_mask = high_floor_mask & ~df_scored['pb_lower_flag']
df_scored['vt_high_floor_bypassed'] = high_floor_mask & df_scored['pb_lower_flag']

df_scored.loc[high_floor_final_mask, 'PRISM_Score'] = (
    df_scored.loc[high_floor_final_mask, 'PRISM_Score'].clip(lower=500)
)

# -------------------------------------------------------------------
# File hash floor
# -------------------------------------------------------------------
df_scored['Severity'] = pd.cut(df_scored['PRISM_Score'], bins=scoring_bins, labels=label_bins, right=False)

file_hash_mask = df_scored['is_file_type']
critical_floor = scoring_bins[3]
df_scored.loc[file_hash_mask, 'PRISM_Score'] = (
    df_scored.loc[file_hash_mask, 'PRISM_Score'].clip(lower=critical_floor)
)
df_scored.loc[file_hash_mask, 'Severity'] = 'critical'

# -------------------------------------------------------------------
# Finalize tag floor band
# -------------------------------------------------------------------
min_tag_floor_score = THREAT_TAG_MIN_FLOOR
tag_floor_band_width = 60

boosted_mask = threat_boost_mask.copy()
below_floor_mask = boosted_mask & (df_scored['PRISM_Score'] < min_tag_floor_score)

if below_floor_mask.any():
    vals = df_scored.loc[below_floor_mask, 'PRISM_Score'].astype(float)
    min_val = vals.min()
    max_val = vals.max()

    if max_val > min_val:
        norm = (vals - min_val) / (max_val - min_val)
        df_scored.loc[below_floor_mask, 'PRISM_Score'] = (
            min_tag_floor_score + norm * tag_floor_band_width
        )
    else:
        df_scored.loc[below_floor_mask, 'PRISM_Score'] = (
            min_tag_floor_score + (tag_floor_band_width / 2.0)
        )

df_scored['PRISM_Score'] = df_scored['PRISM_Score'].clip(upper=1000).round().astype(int)

df_scored['Severity'] = pd.cut(df_scored['PRISM_Score'], bins=scoring_bins, labels=label_bins, right=False)
df_scored.loc[df_scored['is_file_type'], 'Severity'] = 'critical'

df_scored['PRISM_Score_Final'] = df_scored['PRISM_Score'].astype(int)
df_scored['Severity_Final'] = pd.cut(df_scored['PRISM_Score_Final'], bins=scoring_bins, labels=label_bins, right=False)
df_scored.loc[df_scored['is_file_type'], 'Severity_Final'] = 'critical'

# -------------------------------------------------------------------
# Explanation
# -------------------------------------------------------------------
_NAME_MAP = {
    'malicious_raw_score': 'VT malicious (log-scaled)',
    'continuity_raw_score': 'Continuity (indicator type)',
    'tc_raw_rating': 'TC rating',
    'tc_raw_confidence': 'TC confidence',
    'tor_activity_score': 'TOR activity',
    'incidents_events_score': 'Incident/Event association',
    'sources_raw_score': 'Multi-source validation',
    'partner_raw_score': 'Partner coverage bonus',
    'threat_actor_score': 'Threat actor association',
    'cal_raw_score': 'CAL score',
    'tc_threat_raw_score': 'TC threat assessment',
    'first_obs_raw_score': 'Recent first-seen activity',
    'stacked_context_bonus': 'Reinforcing context bonus',
}

def build_explanation(row):
    from datetime import datetime, UTC

    parts = {
        k: float(row.get(k, 0) or 0)
        for k in _NAME_MAP.keys()
    }

    final = row.get('PRISM_Score_Final')
    score = float(final) if pd.notna(final) else float(row.get('PRISM_Score', 0) or 0)
    sev_final = row.get('Severity_Final')
    sev = str(sev_final) if pd.notna(sev_final) else str(row.get('Severity', 'nan'))
    current_date = datetime.now(UTC).strftime('%Y-%m-%d')

    vt_note = (
        f"VT score: {int(row.get('vt_numeric_for_scoring', 0))}."
        if bool(row.get('vt_present', False))
        else "VT score not available (neutral)."
    )

    contrib = sorted(parts.items(), key=lambda kv: abs(kv[1]), reverse=True)[:3]
    driver_bits = [_NAME_MAP.get(k, k) for k, v in contrib if v != 0]
    drivers_text = "; ".join(driver_bits) if driver_bits else "No significant drivers"

    pb_reason = row.get('pb_lower_reason')
    pb_flag = bool(row.get('pb_lower_flag', False))
    pb_mult = float(row.get('pb_base_multiplier', 1.0) or 1.0)
    pb_vt_bypass = bool(row.get('vt_high_floor_bypassed', False))

    if pb_flag:
        pb_note = (
            f"PB lower-start rule applied"
            f"{f' ({pb_reason})' if pd.notna(pb_reason) and str(pb_reason).strip() else ''}; "
            f"base score multiplier {pb_mult:.2f}; incident/event boost suppressed"
            f"{'; VT high floor bypassed' if pb_vt_bypass else ''}."
        )
    else:
        pb_note = "No PB lower-start rule."

    threat_actor_val = row.get('adversary')
    if pd.isna(threat_actor_val) or str(threat_actor_val).strip().lower() in {'none', 'nan', ''}:
        threat_actor_val = row.get('threat_actor')

    actor_sentence = ""
    if threat_actor_val is not None and str(threat_actor_val).strip().lower() not in {'none', 'nan', ''}:
        actor_sentence = f" Associated threat actor(s): {str(threat_actor_val).strip()}."

    botnet_mult = float(row.get('botnet_penalty_multiplier', 1.0))
    scanner_mult = float(row.get('scanner_penalty_multiplier', 1.0))
    fp_cnt = int(row.get('falsePositives', 0) or 0)
    tor_score = float(row.get('tor_activity_score', 0) or 0)
    inc_flag = int(row.get('incidents_events_flag', 0) or 0)

    botnet_note = "Botnet penalty applied." if botnet_mult < 1.0 else "No botnet penalty."
    fp_note = f"{fp_cnt} false positive tag(s)." if fp_cnt > 0 else "No false positive tags."
    scan_note = "Scanner penalty applied." if scanner_mult < 1.0 else "No scanner penalty."
    tor_note = "TOR activity detected." if tor_score > 0 else "No TOR activity."

    if inc_flag == 1 and not pb_flag:
        inc_events = str(row.get('incidents/events', '')).strip()
        inc_note = f"Linked to incident/event: {inc_events}." if inc_events and inc_events.lower() not in {'none', 'nan', ''} else "Linked to incident/event."
    elif inc_flag == 1 and pb_flag:
        inc_note = "Linked to incident/event, but incident/event boost suppressed by PB rule."
    else:
        inc_note = "No incident/event link."

    src_count = int(row.get('sources_count', 1) or 1)
    partner_count = int(row.get('partners_count', 0) or 0)
    stack_count = int(row.get('stacked_context_count', 0) or 0)

    src_note = f"Observed by {src_count} sources." if src_count > 1 else "Single source observation."
    partner_note = f"Observed across {partner_count} partner(s)." if partner_count > 0 else "No partner attribution."
    stack_note = f"{stack_count} reinforcing context signal(s)." if stack_count > 0 else "No reinforcing context stack."

    tagging_reason_val = row.get('Tagging_Boost_Reason')
    tagging_boost_val = bool(row.get('Tagging_Boost', False))
    if pd.notna(tagging_reason_val) and str(tagging_reason_val).strip().lower() not in {'none', 'nan', ''}:
        boost_reason_note = f"Tagging boost: {str(tagging_reason_val).strip()}."
    elif tagging_boost_val:
        boost_reason_note = "Tagging boost: criteria matched."
    else:
        boost_reason_note = ""

    return (
        f"[{current_date}] Severity: {sev}. {vt_note} Contextual Drivers: {drivers_text}. "
        f"{partner_note} {src_note} {stack_note} {pb_note} {botnet_note} {fp_note} "
        f"{scan_note} {tor_note} {inc_note}{actor_sentence} {boost_reason_note} "
        f"Score: {score:.0f}/1000."
    )

df_scored['AI_Adjustment'] = 0
df_scored['Explanation'] = df_scored.apply(build_explanation, axis=1)

# -------------------------------------------------------------------
# Clean up, de-duplicate, and rename for export / reporting
# -------------------------------------------------------------------
if 'indicator' in df_scored.columns:
    df_scored.drop_duplicates(subset='indicator', inplace=True)

column_rename_map = {
    'indicator': 'Indicator',
    'type': 'Indicator Type',
    'lastObserved': 'Last Observed',
    'vt_display': 'VT Display',
    'obs_count': 'Observation Yearly Count',
    'rating': 'ThreatConnect Rating',
    'obs_penalty_multiplier': 'Observation Penalty Multiplier',
    'botnet_flag': 'Botnet Flag',
    'falsePositives': 'False Positives',
    'partners': 'Partners',
    'partners_count': 'Partner Count',
    'sources_count': 'Source Count',
    'adversary': 'Adversary',
    'threat_actor': 'Threat Actor',
    'threat_nation_state': 'Threat Nation State',
    'threat_security_org': 'Threat Security Org',
    'threat_cve_nbr': 'Threat CVE',
    'Tagging_Boost': 'Tagging Boost',
    'Tagging_Boost_Reason': 'Tagging Boost Reason',
    'pb_lower_flag': 'PB Lower Flag',
    'pb_lower_tags': 'PB Lower Tags',
    'pb_lower_reason': 'PB Lower Reason',
    'pb_base_multiplier': 'PB Base Multiplier',
    'vt_high_floor_bypassed': 'VT High Floor Bypassed',
    'calScore': 'CAL Score',
    'threatAssessScore': 'ThreatConnect Score',
    'PRISM_Score': 'PRISM Score',
    'PRISM_Score_Final': 'PRISM Score (Final)',
    'Severity': 'Severity',
    'Severity_Final': 'Severity (Final)',
    'Explanation': 'Explanation',
}

df_scored.rename(columns=column_rename_map, inplace=True)

display_columns = [
    'Indicator', 'Indicator Type', 'Last Observed',
    'VT Display',
    'Observation Yearly Count',
    'ThreatConnect Rating',
    'Observation Penalty Multiplier',
    'Partner Count', 'Source Count',
    'Botnet Flag', 'False Positives', 'Partners', 'incidents/events',
    'ThreatConnect Score', 'CAL Score',
    'Adversary', 'Threat Actor', 'Threat Nation State', 'Threat Security Org', 'Threat CVE',
    'Tagging Boost', 'Tagging Boost Reason',
    'PB Lower Flag', 'PB Lower Tags', 'PB Lower Reason', 'PB Base Multiplier', 'VT High Floor Bypassed',
    'PRISM Score', 'Severity', 'Explanation',
    'PRISM Score (Final)', 'Severity (Final)', 'AI_Adjustment',
]

display_columns = [c for c in display_columns if c in df_scored.columns]

try:
    display(df_scored[display_columns])
except NameError:
    print(df_scored[display_columns].head())

C:\Users\jaskew\AppData\Local\Temp\ipykernel_18984\176778841.py:652: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[576.88259109 568.98785425 576.63967611 568.98785425 560.48582996
 568.98785425 607.61133603 576.63967611 575.30364372 578.09716599
 576.15384615 577.12550607 571.29554656 585.38461538 586.23481781
 569.23076923 563.27935223 599.95951417 600.68825911 567.53036437
 567.53036437 589.75708502 568.98785425 560.48582996 576.63967611
 576.15384615 584.65587045 563.27935223 568.98785425 576.63967611
 582.59109312 589.87854251 585.1417004  585.38461538 586.11336032
 584.89878543 563.6437247  585.50607287 578.70445344 568.13765182
 576.63967611 568.13765182 568.13765182 592.67206478 590.48582996
 578.70445344 564.37246964 577.85425101 563.88663968 568.98785425
 568.98785425 568.98785425 609.79757085 586.3562753  582.59109312
 562.91497976 564.12955466 577.85425101 562.91497976 568.13765182
 586.23481

,Indicator,Indicator Type,Last Observed,VT Display,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Partner Count,Source Count,Botnet Flag,...,PB Lower Tags,PB Lower Reason,PB Base Multiplier,VT High Floor Bypassed,PRISM Score,Severity,Explanation,PRISM Score (Final),Severity (Final),AI_Adjustment
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,File,2026-01-08 00:00:00+00:00,No VT Score,2,3.0,0.999890,0,1,1,...,[],None,1.00,False,1000,critical,[2026-05-03] Severity: critical. VT score not ...,1000,critical,0
1,1-you.njalla.no,Host,2026-04-01 00:00:00+00:00,No VT Score,37,5.0,0.997973,2,1,0,...,[],None,1.00,False,577,high,[2026-05-03] Severity: high. VT score not avai...,577,high,0
2,1.192.18.4,Address,2026-04-21 00:00:00+00:00,6.0,19,3.0,0.998959,5,1,0,...,[SOAR Indicator PB],pb_tag:SOAR Indicator PB,0.45,False,158,low,[2026-05-03] Severity: low. VT score: 6. Conte...,158,low,0
3,1.20.169.90,Address,2026-03-29 00:00:00+00:00,0.0,3,5.0,0.999836,1,1,0,...,[],None,1.00,False,569,high,[2026-05-03] Severity: high. VT score: 0. Cont...,569,high,0
4,1.234.83.26,Address,2026-02-14 00:00:00+00:00,3.0,43,3.0,0.997644,0,1,0,...,[SOAR Indicator PB],pb_tag:SOAR Indicator PB,0.45,False,88,low,[2026-05-03] Severity: low. VT score: 3. Conte...,88,low,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3464,www.sourcefiles.org,Host,2025-08-13 00:00:00+00:00,No VT Score,1,5.0,0.999945,1,1,0,...,[],None,1.00,False,91,low,[2026-05-03] Severity: low. VT score not avail...,91,low,0
3465,www.sthda.com,Host,2026-03-03 00:00:00+00:00,No VT Score,194,4.0,0.989370,8,1,0,...,[],None,1.00,False,153,low,[2026-05-03] Severity: low. VT score not avail...,153,low,0
3466,yourpensionmeeting.com,Host,2026-03-07 00:00:00+00:00,No VT Score,222,3.0,0.987836,4,1,0,...,[],None,1.00,False,143,low,[2026-05-03] Severity: low. VT score not avail...,143,low,0
3467,zerogov.com,Host,2025-11-12 00:00:00+00:00,No VT Score,1,5.0,0.999945,1,1,0,...,[],None,1.00,False,97,low,[2026-05-03] Severity: low. VT score not avail...,97,low,0


In [120]:
import numpy as np
import pandas as pd

# --- Rule-layer importance derived from df_scored ---
# These are the additive rule components already stored per row
rule_components = [
    'malicious_raw_score',
    'continuity_raw_score',
    'tc_raw_rating',
    'tc_raw_confidence',
    'tor_activity_score',
    'incidents_events_score',
    'sources_raw_score',
    'partner_raw_score',
    'threat_actor_score',
    'cal_raw_score',
    'tc_threat_raw_score',
    'first_obs_raw_score',
]

existing_rule_components = [c for c in rule_components if c in df_scored.columns]

# Per-row absolute contributions
rule_abs = df_scored[existing_rule_components].abs()
row_sums = rule_abs.sum(axis=1).replace(0, np.nan)
rule_rel = rule_abs.div(row_sums, axis=0)

# Global average relative importance across all rows
rule_global_importance = rule_rel.mean().sort_values(ascending=False)

print("Average relative importance of rule components (across all indicators):")
display(rule_global_importance)

# Optional: show top drivers for a single example indicator by index
example_idx = 0  # change to inspect a different row
print(f"\nTop rule drivers for index {example_idx} (absolute contribution):")
example_contrib = rule_abs.loc[example_idx].sort_values(ascending=False)
display(example_contrib)

Average relative importance of rule components (across all indicators):


malicious_raw_score       0.395640
tor_activity_score        0.218142
threat_actor_score        0.116871
partner_raw_score         0.097031
continuity_raw_score      0.062812
incidents_events_score    0.045355
cal_raw_score             0.028161
sources_raw_score         0.014977
tc_threat_raw_score       0.012442
tc_raw_confidence         0.006553
tc_raw_rating             0.001308
first_obs_raw_score       0.000709
dtype: float64


Top rule drivers for index 0 (absolute contribution):


continuity_raw_score      900.000
incidents_events_score      8.000
cal_raw_score               2.035
malicious_raw_score         0.000
tc_raw_confidence           0.000
tc_raw_rating               0.000
sources_raw_score           0.000
tor_activity_score          0.000
partner_raw_score           0.000
threat_actor_score          0.000
tc_threat_raw_score         0.000
first_obs_raw_score         0.000
Name: 0, dtype: float64

In [121]:
# Save the DataFrame to Excel with only the columns displayed in cell 14
import os
import pandas as pd
from datetime import datetime
from openpyxl.styles import PatternFill

output_dir = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores"
os.makedirs(output_dir, exist_ok=True)

# Create filename
excel_filename = "Threat_Assessment_Scores.xlsx"
excel_path = os.path.join(output_dir, excel_filename)

# Select only the columns to save (using renamed column names)
columns_to_save = ['Indicator', 'Last Observed', 'Indicator Type', 'VirusTotal Malicious Score', 
                   'Observation Yearly Count', 'ThreatConnect Rating', 'Observation Penalty Multiplier',
                   'Botnet Flag', 'False Positives', 'Partners', 'incidents/events', 'Threat Actor',
                   'Threat Nation State', 'Threat Security Org', 'Threat CVE', 'Tagging Boost', 'Tagging Boost Reason',
                   'CAL Score', 'ThreatConnect Score', 'PRISM Score', 'Severity', 'Explanation']

# Make a copy with only selected columns (only include columns that exist)
columns_to_save = [col for col in columns_to_save if col in df_scored.columns]
df_export = df_scored[columns_to_save].copy()

# Use 'PRISM Score (Final)' values for the 'PRISM Score' column in Excel
if 'PRISM Score (Final)' in df_scored.columns:
    df_export['PRISM Score'] = df_scored['PRISM Score (Final)']
    # Also update Severity to use Severity (Final) if available
    if 'Severity (Final)' in df_scored.columns:
        df_export['Severity'] = df_scored['Severity (Final)']

# Convert timezone-aware datetime columns to timezone-naive for Excel compatibility
for col in df_export.columns:
    if pd.api.types.is_datetime64_any_dtype(df_export[col]):
        # Check if the column has timezone info
        if df_export[col].dt.tz is not None:
            # Convert to UTC first, then remove timezone info to make it timezone-naive
            df_export[col] = df_export[col].dt.tz_convert('UTC').dt.tz_localize(None)

# Check if file exists and read existing data
if os.path.exists(excel_path):
    df_existing = pd.read_excel(excel_path, engine='openpyxl')
    
    # Normalize existing columns to the latest schema (older files may still use raw names)
    rename_map_existing = {
        old: new for old, new in column_rename_map.items()
        if old in df_existing.columns and new not in df_existing.columns
    }
    if rename_map_existing:
        df_existing.rename(columns=rename_map_existing, inplace=True)
    
    # Add missing columns to existing dataframe with default values
    for col in columns_to_save:
        if col not in df_existing.columns:
            # Set appropriate default values based on column type
            if col == 'Last Observed':
                df_existing[col] = pd.NaT
            elif col in ['VirusTotal Malicious Score', 'Observation Yearly Count', 'ThreatConnect Rating', 
                        'Observation Penalty Multiplier', 'Botnet Flag', 'False Positives', 'PRISM Score', 'ThreatConnect Score']:
                df_existing[col] = 0
            elif col == 'Severity':
                df_existing[col] = ''
            else:
                df_existing[col] = ''
    
    # Ensure both dataframes have the same columns in the same order
    df_existing = df_existing[columns_to_save].copy()
    
    # Count how many indicators will be updated vs added
    existing_indicators = set(df_existing['Indicator'].values)
    new_indicators = set(df_export['Indicator'].values)
    
    indicators_to_add = len(new_indicators - existing_indicators)
    indicators_to_check = existing_indicators & new_indicators
    
    # Find indicators that have actually changed
    indicators_to_update = []
    indicators_unchanged = []
    
    # Set indicator as index for comparison
    df_existing_idx = df_existing.set_index('Indicator').sort_index()
    df_export_idx = df_export.set_index('Indicator').sort_index()
    
    for indicator in indicators_to_check:
        # Compare the rows (excluding the index/indicator column)
        existing_row = df_existing_idx.loc[indicator]
        new_row = df_export_idx.loc[indicator]
        
        # Check if any values have changed
        if not existing_row.equals(new_row):
            indicators_to_update.append(indicator)
        else:
            indicators_unchanged.append(indicator)
    
    # Build the combined dataframe: keep existing unchanged records, update changed ones, add new ones
    # Start with existing records that are unchanged
    df_unchanged = df_existing[df_existing['Indicator'].isin(indicators_unchanged)].copy()
    
    # Add updated records (from new data)
    df_updated = df_export[df_export['Indicator'].isin(indicators_to_update)].copy()
    
    # Add new records (not in existing)
    df_new = df_export[df_export['Indicator'].isin(new_indicators - existing_indicators)].copy()
    
    # Add existing records that are not in new data (preserve historical data)
    df_preserved = df_existing[~df_existing['Indicator'].isin(new_indicators)].copy()
    
    # Combine all parts
    df_combined = pd.concat([df_unchanged, df_updated, df_new, df_preserved], ignore_index=True)
    
    # Final check: remove any duplicates (shouldn't happen, but safety check)
    df_combined = df_combined.drop_duplicates(subset='Indicator', keep='last')
    
    print(f"Existing indicators in new data: {len(indicators_to_check)}")
    print(f"Added (new) indicators: {indicators_to_add}")
    print(f"Updated existing indicators: {len(indicators_to_update)}")
    print(f"Total indicators in sheet: {len(df_combined)}")
else:
    df_combined = df_export.copy()
    # Remove any duplicates in the new data itself
    df_combined = df_combined.drop_duplicates(subset='Indicator', keep='last')
    print(f"Created new file with {len(df_combined)} indicators")

# Read existing Complete History if it exists (to preserve it when rewriting file)
df_complete_history = None
if os.path.exists(excel_path):
    try:
        df_complete_history = pd.read_excel(excel_path, sheet_name='Complete History', engine='openpyxl')
        print(f"Preserving existing Complete History with {len(df_complete_history)} records")
    except:
        pass  # Sheet doesn't exist yet, that's okay

# Save to Excel with severity-based highlighting
with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    df_combined.to_excel(writer, index=False, sheet_name='PRISM Scores')

    # Create and write comparison sheet
    if 'ThreatConnect Score' in df_combined.columns and 'PRISM Score' in df_combined.columns:
        comp_cols = ['Indicator', 'ThreatConnect Score', 'PRISM Score']
        df_comp = df_combined[comp_cols].copy()
        # Ensure numeric
        df_comp['ThreatConnect Score'] = pd.to_numeric(df_comp['ThreatConnect Score'], errors='coerce').fillna(0)
        df_comp['PRISM Score'] = pd.to_numeric(df_comp['PRISM Score'], errors='coerce').fillna(0)
        df_comp['Difference'] = df_comp['PRISM Score'] - df_comp['ThreatConnect Score']
        df_comp.to_excel(writer, index=False, sheet_name='Score Comparison')

    # Write Complete History if it exists (preserve it)
    if df_complete_history is not None:
        df_complete_history.to_excel(writer, index=False, sheet_name='Complete History')

    workbook = writer.book
    worksheet = writer.sheets['PRISM Scores']

    

    # Define fills for each severity
    fills = {
        'low': PatternFill(start_color='83de85', end_color='83de85', fill_type='solid'),     # light green
        'medium': PatternFill(start_color='eef084', end_color='eef084', fill_type='solid'),  # light yellow
        'high': PatternFill(start_color='f29953', end_color='f29953', fill_type='solid'),    # light orange
        'critical': PatternFill(start_color='e83f3f', end_color='e83f3f', fill_type='solid') # light red
    }

    for row_idx, severity in enumerate(df_combined['Severity'], start=2):  # start=2 to skip header
        fill = fills.get(str(severity).lower())
        if fill:
            for col_idx in range(1, len(df_combined.columns) + 1):
                worksheet.cell(row=row_idx, column=col_idx).fill = fill

print(f"Saved {len(df_combined)} total indicators with PRISM scoring results to {excel_path}")

Existing indicators in new data: 3458
Added (new) indicators: 11
Updated existing indicators: 3458
Total indicators in sheet: 3772
Preserving existing Complete History with 15415 records
Saved 3772 total indicators with PRISM scoring results to Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx


In [122]:
import os
from datetime import datetime, UTC
import pandas as pd
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl import load_workbook

# ============================================================================
# INDICATOR SCORING HISTORY - Similar to Credit Report History
# Maintains a time-series record of all indicator scores and explanations in Excel
# ============================================================================

# Define history file path - using the same file as main Prioritized Risk and Indicator Severity Model scores
history_dir = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores"
os.makedirs(history_dir, exist_ok=True)

# Use the main Threat_Assessment_Scores.xlsx file instead of separate file
history_file = os.path.join(history_dir, "Threat_Assessment_Scores.xlsx")

# Record the current scoring run (using timezone-aware UTC)
run_timestamp = datetime.now(UTC).strftime('%Y-%m-%d %H:%M:%S')

# Define columns to track - only essential fields
history_columns = [
    'Scoring Date',
    'Indicator',
    'Indicator Type',
    'PRISM Score',
    'Severity',
    'Explanation'
]

# Map dataframe columns to history columns
column_mapping = {
    'Indicator': 'Indicator',
    'Indicator Type': 'Indicator Type',
    'PRISM Score (Final)': 'PRISM Score',
    'Severity (Final)': 'Severity',
    'Explanation': 'Explanation'
}

# Build history data for this run
history_data = []
for idx, row in df_scored.iterrows():
    history_row = {}
    for df_col, hist_col in column_mapping.items():
        if df_col in df_scored.columns:
            value = row[df_col]
            # Handle datetime/timestamp conversions
            if pd.api.types.is_datetime64_any_dtype(type(value)):
                history_row[hist_col] = str(value) if pd.notna(value) else ''
            elif pd.isna(value):
                history_row[hist_col] = ''
            else:
                history_row[hist_col] = value
    
    history_row['Scoring Date'] = run_timestamp
    history_data.append(history_row)

df_history_current = pd.DataFrame(history_data)

# Reorder columns to match history_columns order
df_history_current = df_history_current[[col for col in history_columns if col in df_history_current.columns]]

# Load existing history from the Excel file if it exists
if os.path.exists(history_file):
    try:
        # Try to read the Complete History sheet
        df_history_all = pd.read_excel(history_file, sheet_name='Complete History', engine='openpyxl')
        # Convert any timezone-aware columns to timezone-naive BEFORE concatenation
        for col in df_history_all.columns:
            if pd.api.types.is_datetime64_any_dtype(df_history_all[col]):
                if hasattr(df_history_all[col].dtype, 'tz') and df_history_all[col].dtype.tz is not None:
                    df_history_all[col] = df_history_all[col].dt.tz_convert('UTC').dt.tz_localize(None)
        
        # Append new records to existing history - preserve all historical records
        # Exclude completely empty/all-NA columns before concatenation to avoid deprecation warning
        df_history_all = df_history_all.dropna(axis=1, how='all')
        df_history_current = df_history_current.dropna(axis=1, how='all')
        df_history_all = pd.concat([df_history_all, df_history_current], ignore_index=True)
    except:
        # If Complete History sheet doesn't exist, start fresh
        df_history_all = df_history_current.copy()
else:
    df_history_all = df_history_current.copy()

# Remove duplicates only if the same indicator was scored on the same date (keep most recent)
# Extract date only (without time) for duplicate detection to prevent time-based duplicates
df_history_all['Scoring Date Only'] = pd.to_datetime(df_history_all['Scoring Date']).dt.date
df_history_all = df_history_all.drop_duplicates(subset=['Indicator', 'Scoring Date Only'], keep='last')
# Remove the temporary column
df_history_all = df_history_all.drop(columns=['Scoring Date Only'])

# Sort by Indicator and Scoring Date for easier review
df_history_all = df_history_all.sort_values(['Indicator', 'Scoring Date'], ascending=[True, False])

# Convert timezone-aware datetime columns to timezone-naive for Excel compatibility
df_history_all = df_history_all.copy()  # Make a copy to avoid SettingWithCopyWarning
for col in df_history_all.columns:
    if pd.api.types.is_datetime64_any_dtype(df_history_all[col]):
        # Check if column has timezone info
        if hasattr(df_history_all[col].dtype, 'tz') and df_history_all[col].dtype.tz is not None:
            df_history_all[col] = df_history_all[col].dt.tz_convert('UTC').dt.tz_localize(None)
        elif df_history_all[col].dt.tz is not None:
            df_history_all[col] = df_history_all[col].dt.tz_convert('UTC').dt.tz_localize(None)

# Reorder columns in complete history to match desired order
df_history_all = df_history_all[[col for col in history_columns if col in df_history_all.columns]]

# Read existing sheets to preserve them
existing_sheets = {}
if os.path.exists(history_file):
    try:
        excel_file = pd.ExcelFile(history_file, engine='openpyxl')
        for sheet_name in excel_file.sheet_names:
            if sheet_name not in ['Complete History', 'Latest Scores']:
                existing_sheets[sheet_name] = pd.read_excel(history_file, sheet_name=sheet_name, engine='openpyxl')
        excel_file.close()
    except Exception as e:
        print(f"Warning: Could not read existing sheets: {e}")

# Now write everything back - this mimics the pattern from the working code
with pd.ExcelWriter(history_file, engine='openpyxl') as writer:
    # Write existing sheets first (PRISM Scores and Score Comparison)
    for sheet_name, sheet_df in existing_sheets.items():
        sheet_df.to_excel(writer, index=False, sheet_name=sheet_name)
        
        # If this is the PRISM Scores sheet, reapply the severity formatting
        if sheet_name == 'PRISM Scores' and 'Severity' in sheet_df.columns:
            worksheet = writer.sheets['PRISM Scores']
            
            # Define fills for each severity (same as original code)
            fills = {
                'low': PatternFill(start_color='83de85', end_color='83de85', fill_type='solid'),     # light green
                'medium': PatternFill(start_color='eef084', end_color='eef084', fill_type='solid'),  # light yellow
                'high': PatternFill(start_color='f29953', end_color='f29953', fill_type='solid'),    # light orange
                'critical': PatternFill(start_color='e83f3f', end_color='e83f3f', fill_type='solid') # light red
            }
            
            for row_idx, severity in enumerate(sheet_df['Severity'], start=2):  # start=2 to skip header
                fill = fills.get(str(severity).lower())
                if fill:
                    for col_idx in range(1, len(sheet_df.columns) + 1):
                        worksheet.cell(row=row_idx, column=col_idx).fill = fill
    
    # Write Complete History
    df_history_all.to_excel(writer, sheet_name='Complete History', index=False)
    
    # Format "Complete History" sheet
    ws_history = writer.sheets['Complete History']
    ws_history.column_dimensions['A'].width = 20  # Scoring Date
    ws_history.column_dimensions['B'].width = 20  # Indicator
    ws_history.column_dimensions['C'].width = 15  # Indicator Type
    ws_history.column_dimensions['D'].width = 18  # PRISM Score
    ws_history.column_dimensions['E'].width = 12  # Severity
    ws_history.column_dimensions['F'].width = 60  # Explanation
    
    # Add header formatting to Complete History
    for cell in ws_history[1]:
        cell.font = Font(bold=True, color='FFFFFF')
        cell.fill = PatternFill(start_color='366092', end_color='366092', fill_type='solid')
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)

print(f"✓ Scoring history updated in: {history_file}")
print(f"✓ Excel file now contains:")
print(f"  - PRISM Scores (current scores with all details and severity formatting)")
print(f"  - Score Comparison (PRISM vs ThreatConnect)")
print(f"  - Complete History (time-series of all historical scores)")
print(f"✓ Total indicators in history: {len(df_history_all['Indicator'].unique())}")
print(f"✓ Total scoring records: {len(df_history_all)}")

# ============================================================================
# HISTORY LOOKUP FUNCTIONS - Query past scores like a credit report
# ============================================================================

def get_indicator_score_history(indicator):
    """
    Retrieve the complete scoring history for a single indicator.
    Similar to pulling a credit report for one person.
    """
    history = df_history_all[df_history_all['Indicator'] == indicator].copy()
    if history.empty:
        return None
    
    return history.sort_values('Scoring Date', ascending=False)

def get_score_changes_since(days_ago=7):
    """
    Find all indicators whose scores have changed in the last N days.
    Useful for identifying trending threats.
    """
    cutoff_date = pd.Timestamp.utcnow() - pd.Timedelta(days=days_ago)
    
    changed_indicators = []
    for indicator in df_history_all['Indicator'].unique():
        ind_history = df_history_all[df_history_all['Indicator'] == indicator].sort_values('Scoring Date', ascending=False)
        
        if len(ind_history) > 1:
            recent = ind_history[pd.to_datetime(ind_history['Scoring Date']) >= cutoff_date]
            
            if len(recent) > 1:
                old_score = recent.iloc[-1]['PRISM Score']
                new_score = recent.iloc[0]['PRISM Score']
                
                if pd.notna(old_score) and pd.notna(new_score) and old_score != new_score:
                    changed_indicators.append({
                        'Indicator': indicator,
                        'Type': recent.iloc[0]['Indicator Type'],
                        'Previous Score': old_score,
                        'Current Score': new_score,
                        'Change': new_score - old_score,
                        'Last Updated': recent.iloc[0]['Scoring Date']
                    })
    
    if changed_indicators:
        return pd.DataFrame(changed_indicators).sort_values('Change', ascending=False, key=abs)
    else:
        return pd.DataFrame()

# Display summary statistics
print("\n" + "="*70)
print("SCORING HISTORY SUMMARY")
print("="*70)

# Count indicators with multiple records
multi_record = df_history_all.groupby('Indicator').size()
indicators_with_history = sum(multi_record > 1)
print(f"Indicators with multiple scoring records: {indicators_with_history}")

# Show example: lookup a specific indicator if any exist
if not df_scored.empty:
    example_indicator = df_scored['Indicator'].iloc[0]
    example_history = get_indicator_score_history(example_indicator)
    if example_history is not None:
        print(f"\nExample - Scoring history for {example_indicator}:")
        # Only show columns that exist
        example_cols = ['Scoring Date']
        if 'PRISM Score' in example_history.columns:
            example_cols.append('PRISM Score')
        if 'Severity' in example_history.columns:
            example_cols.append('Severity')
        if 'Explanation' in example_history.columns:
            example_cols.append('Explanation')
        print(example_history[example_cols].head())

print(f"\n✓ History functions available:")
print(f"  - get_indicator_score_history(indicator)  # Get full history for one indicator")
print(f"  - get_score_changes_since(days_ago=7)      # Find indicators with score changes")

✓ Scoring history updated in: Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx
✓ Excel file now contains:
  - PRISM Scores (current scores with all details and severity formatting)
  - Score Comparison (PRISM vs ThreatConnect)
  - Complete History (time-series of all historical scores)
✓ Total indicators in history: 3772
✓ Total scoring records: 18884

SCORING HISTORY SUMMARY
Indicators with multiple scoring records: 3465

Example - Scoring history for 0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D140CD5A821FF08ACFAC:
              Scoring Date  PRISM Score  Severity  \
15415  2026-05-03 18:28:41         1000  critical   
0      2026-05-01 14:12:56         1000  critical   
1      2026-04-30 14:58:18         1000  critical   
2      2026-04-29 14:20:13         1000  critical   
3      2026-04-20 11:47:23         1000  critical   

                                             Explanation  
15415  [2026-05-03] Severity: critical. VT score not ...  
0     

import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

# -------------------------------------------------------------------
# Setup
# -------------------------------------------------------------------
df_plot = df_scored.copy()

score_col = 'PRISM Score (Final)' if 'PRISM Score (Final)' in df_plot.columns else 'PRISM_Score_Final'
sev_col = 'Severity (Final)' if 'Severity (Final)' in df_plot.columns else 'Severity_Final'
boost_col = 'Tagging Boost' if 'Tagging Boost' in df_plot.columns else 'Tagging_Boost'
boost_reason_col = 'Tagging Boost Reason' if 'Tagging Boost Reason' in df_plot.columns else 'Tagging_Boost_Reason'
pb_col = 'PB Lower Flag' if 'PB Lower Flag' in df_plot.columns else 'pb_lower_flag'
pb_reason_col = 'PB Lower Reason' if 'PB Lower Reason' in df_plot.columns else 'pb_lower_reason'
vt_bypass_col = 'VT High Floor Bypassed' if 'VT High Floor Bypassed' in df_plot.columns else 'vt_high_floor_bypassed'

df_plot[score_col] = pd.to_numeric(df_plot[score_col], errors='coerce')
df_plot[boost_col] = df_plot[boost_col].fillna(False).astype(bool) if boost_col in df_plot.columns else False
df_plot[pb_col] = df_plot[pb_col].fillna(False).astype(bool) if pb_col in df_plot.columns else False

if vt_bypass_col in df_plot.columns:
    df_plot[vt_bypass_col] = df_plot[vt_bypass_col].fillna(False).astype(bool)
else:
    df_plot[vt_bypass_col] = False

report_dir = "prism_score_distribution_report"
os.makedirs(report_dir, exist_ok=True)
pdf_path = os.path.join(report_dir, "PRISM_Tag_Rule_Validation_Report.pdf")

# -------------------------------------------------------------------
# Summary table
# -------------------------------------------------------------------
summary = pd.DataFrame({
    'Metric': [
        'Total indicators',
        'Tagging boost indicators',
        'PB lower-rule indicators',
        'Both tagging boost and PB lower-rule',
        'VT high floor bypassed by PB rule',
        'Average final score',
        'Median final score',
        'Average score - tagging boost',
        'Median score - tagging boost',
        'Average score - non-boosted',
        'Median score - non-boosted',
        'Average score - PB lower rule',
        'Median score - PB lower rule',
        'Average score - non-PB',
        'Median score - non-PB',
    ],
    'Value': [
        len(df_plot),
        int(df_plot[boost_col].sum()),
        int(df_plot[pb_col].sum()),
        int((df_plot[boost_col] & df_plot[pb_col]).sum()),
        int(df_plot[vt_bypass_col].sum()),
        round(df_plot[score_col].mean(), 2),
        round(df_plot[score_col].median(), 2),
        round(df_plot.loc[df_plot[boost_col], score_col].mean(), 2),
        round(df_plot.loc[df_plot[boost_col], score_col].median(), 2),
        round(df_plot.loc[~df_plot[boost_col], score_col].mean(), 2),
        round(df_plot.loc[~df_plot[boost_col], score_col].median(), 2),
        round(df_plot.loc[df_plot[pb_col], score_col].mean(), 2),
        round(df_plot.loc[df_plot[pb_col], score_col].median(), 2),
        round(df_plot.loc[~df_plot[pb_col], score_col].mean(), 2),
        round(df_plot.loc[~df_plot[pb_col], score_col].median(), 2),
    ]
})

# -------------------------------------------------------------------
# Chart helpers
# -------------------------------------------------------------------
def save_current_fig(pdf=None):
    plt.tight_layout()
    if pdf is not None:
        pdf.savefig(plt.gcf(), bbox_inches='tight')
    plt.show()
    plt.close()

def add_summary_page(pdf, summary_df):
    fig, ax = plt.subplots(figsize=(11, 8.5))
    ax.axis('off')

    ax.text(
        0.5, 0.96,
        "PRISM Tag Rule Validation Report",
        ha='center',
        va='top',
        fontsize=18,
        fontweight='bold'
    )

    ax.text(
        0.5, 0.91,
        "Purpose: confirm PB scanning tags suppressed scores and tagging boost rules boosted scores.",
        ha='center',
        va='top',
        fontsize=10
    )

    table = ax.table(
        cellText=summary_df.values.tolist(),
        colLabels=summary_df.columns,
        cellLoc='left',
        colLoc='left',
        loc='center',
        bbox=[0.08, 0.08, 0.84, 0.76]
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.2)

    pdf.savefig(fig, bbox_inches='tight')
    plt.close(fig)

# -------------------------------------------------------------------
# Build charts and PDF
# -------------------------------------------------------------------
with PdfPages(pdf_path) as pdf:
    add_summary_page(pdf, summary)

    # 1. Overall score distribution
    fig, ax = plt.subplots(figsize=(10, 5))
    n, bins, patches = ax.hist(df_plot[score_col].dropna(), bins=30, alpha=0.7, edgecolor='black')
    ax.set_title('PRISM Score Distribution')
    ax.set_xlabel('PRISM Score')
    ax.set_ylabel('Indicator Count')

    for count, x in zip(n, bins):
        if count > 0:
            ax.text(x + (bins[1] - bins[0]) / 2, count, int(count), ha='center', va='bottom', fontsize=8)

    save_current_fig(pdf)

    # 2. Severity distribution
    severity_order = ['low', 'medium', 'high', 'critical']
    sev_counts = (
        df_plot[sev_col]
        .astype(str)
        .str.lower()
        .value_counts()
        .reindex(severity_order)
        .fillna(0)
    )

    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(severity_order, sev_counts.values, color=['#8dd3c7', '#ffffb3', '#fb8072', '#bc80bd'])

    for bar, count in zip(bars, sev_counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), int(count), ha='center', va='bottom', fontsize=10)

    ax.set_title('Severity Distribution')
    ax.set_xlabel('Severity')
    ax.set_ylabel('Indicator Count')
    ax.legend(['Counts'], loc='upper right')
    ax.set_ylim(0, max(sev_counts.values) * 1.15 if max(sev_counts.values) > 0 else 1)
    plt.xticks(rotation=0)

    save_current_fig(pdf)

    # 3. Tagging boost vs non-boosted
    fig, ax = plt.subplots(figsize=(10, 5))
    bins = 30
    non_boosted = df_plot.loc[~df_plot[boost_col], score_col].dropna()
    boosted = df_plot.loc[df_plot[boost_col], score_col].dropna()

    ax.hist(non_boosted, bins=bins, alpha=0.6, label=f'No Tagging Boost (n={len(non_boosted)})', edgecolor='black')
    ax.hist(boosted, bins=bins, alpha=0.6, label=f'Tagging Boost (n={len(boosted)})', edgecolor='black')

    ax.set_title('Score Distribution: Tagging Boost vs Non-Boosted')
    ax.set_xlabel('PRISM Score')
    ax.set_ylabel('Indicator Count')
    ax.legend()

    save_current_fig(pdf)

    # 4. PB lower rule vs non-PB
    fig, ax = plt.subplots(figsize=(10, 5))
    no_pb = df_plot.loc[~df_plot[pb_col], score_col].dropna()
    with_pb = df_plot.loc[df_plot[pb_col], score_col].dropna()

    ax.hist(no_pb, bins=bins, alpha=0.6, label=f'No PB Lower Rule (n={len(no_pb)})', edgecolor='black')
    ax.hist(with_pb, bins=bins, alpha=0.6, label=f'PB Lower Rule (n={len(with_pb)})', edgecolor='black')

    ax.set_title('Score Distribution: PB Lower Rule vs Non-PB')
    ax.set_xlabel('PRISM Score')
    ax.set_ylabel('Indicator Count')
    ax.legend()

    save_current_fig(pdf)

    # 5. Tagging boost reason distribution
    if boost_reason_col in df_plot.columns:
        reason_counts = (
            df_plot.loc[df_plot[boost_col], boost_reason_col]
            .fillna('unknown')
            .value_counts()
            .head(20)
        )

        if len(reason_counts) > 0:
            fig, ax = plt.subplots(figsize=(12, 6))
            reason_counts_sorted = reason_counts.sort_values()
            reason_counts_sorted.plot(kind='barh', ax=ax, color='#6495ed', edgecolor='black')
            ax.set_title('Top Tagging Boost Reasons')
            ax.set_xlabel('Indicator Count')
            ax.set_ylabel('Tagging Boost Reason')
            ax.grid(axis='x', linestyle='--', alpha=0.5)

            for patch, count in zip(ax.patches, reason_counts_sorted):
                ax.text(
                    patch.get_width(),
                    patch.get_y() + patch.get_height() / 2,
                    int(count),
                    ha='left',
                    va='center',
                    fontsize=9,
                    color='black'
                )

            save_current_fig(pdf)

    # 6. PB lower reason distribution
    if pb_reason_col in df_plot.columns:
        pb_reason_counts = (
            df_plot.loc[df_plot[pb_col], pb_reason_col]
            .fillna('unknown')
            .value_counts()
        )

        if len(pb_reason_counts) > 0:
            fig, ax = plt.subplots(figsize=(10, 5))
            pb_reason_counts_sorted = pb_reason_counts.sort_values()
            pb_reason_counts_sorted.plot(kind='barh', ax=ax, color='#ffcc99', edgecolor='black')
            ax.set_title('PB Lower Rule Reasons')
            ax.set_xlabel('Indicator Count')
            ax.set_ylabel('PB Lower Reason')
            ax.grid(axis='x', linestyle='--', alpha=0.5)

            for patch, count in zip(ax.patches, pb_reason_counts_sorted):
                ax.text(
                    patch.get_width(),
                    patch.get_y() + patch.get_height() / 2,
                    int(count),
                    ha='left',
                    va='center',
                    fontsize=9,
                    color='black'
                )

            save_current_fig(pdf)

print(f"PDF report created: {pdf_path}")
display(summary)